# CohAlign — predicting noise-induced gradient degradation in equivariant quantum neural networks


> **v1.8.1 hotfix.** The best-effort notebook hash in the summary cell no
> longer falls back to a recursive search of the mounted Drive, which on a
> large Drive could run for many minutes and appear to hang. If the notebook
> file is not in the working directory the manifest records `not_located`.
> No caches are affected; re-running the summary cell alone updates the
> metadata.

> **v1.8 (this revision).** Adds **Phase K, the parameter-resolved validation
> map** requested in review, comparing the generator-derived susceptibility
> with a matched-ensemble finite-difference limit across the most active
> parameter locations at depth `pm_L` (default 4, where the light cone breaks
> the shallow-geometry degeneracy) for the channels in `pm_channels`, with
> inactive locations skipped explicitly. A new figure `fig6_param_map.png` is
> rendered from `phaseK_param_map.csv`. Also fixes a v1.7.1 bug in which the
> degenerate zero-gradient early return of the response estimator omitted the
> recorded bootstrap fields and crashed the audit on inactive parameters.
> **No caches need deleting for the upgrade from a completed v1.7.1 run.**

**Pipeline notebook.** This notebook is the complete, self-contained
computational pipeline for the CohAlign study. Executed top to bottom it
regenerates every table, CSV and figure of the paper from scratch: no external
data, no other repositories, and no dependencies beyond `numpy`, `pandas` and
`matplotlib`.

## What this study contributes

In noisy U(1)-equivariant brickwork QNNs, active-gradient degradation at
first order in the noise strength is governed not by how fast a channel
contracts sector coherence in the worst case, but by how fast it contracts
the specific coherence the readout can see. This project makes that
distinction quantitative through two generator-level rates on the in-sector
coherence block: the **worst-case sector coherence rate**
$\lambda_{\mathrm{coh}}$ and the **readout-visible aligned rate**
$\lambda_{\mathrm{vis}}$, the contraction speed of the gradient mode itself.

**This notebook makes the aligned rate computable a priori and validates the
computation.** It contributes:

1. **Two computable, generator-level estimators** of the aligned rate
   (module `cohalign_rates`):
   the *operator-level quadratic estimator* $\lambda_{\mathrm{vis}}^{\mathrm{op}}$
   (the literal computable form of the defining Rayleigh quotient, with the
   readout projection and slot geometry built in), and the
   *response-weighted layer-resolved estimator* — exact first-order
   degradation rates $r_\ell$, one per noise slot, whose sum predicts the
   measured degradation *a priori*.
2. **A five-rung validation ladder** (Phases B–F below) from exact isotropic
   recovery, through the zero-alignment control *predicted from the generator
   alone*, to family-wide predicted-vs-measured alignment across nine
   channels.
3. **A predictor upgrade** (Phase G): the pooled degradation regression refit
   with the *a priori* aligned rate replacing the post-hoc weighted rate.
4. **A noise-aware audit toolkit**
   (`audit_channel(...)`: one call from characterised channel to
   $\lambda_{\mathrm{coh}}$, $\lambda_{\mathrm{vis}}$, per-slot rates and the
   predicted alignment ratio $\omega$), with a worked example on a
   user-supplied channel.

## How to run

```bash
COHALIGN_MODE=smoke   jupyter nbconvert --to notebook --execute cohalign_pipeline.ipynb   # minutes
COHALIGN_MODE=paper   ...                                                                  # hours
COHALIGN_MODE=figures ...   # re-render figures from cached CSVs only
```

Set the mode in the configuration cell or via the `COHALIGN_MODE` environment
variable. On **Google Colab** the notebook mounts Drive and writes every CSV,
figure and metadata file to
`MyDrive/Colab Notebooks/TEMP/Quantum Neural Networks/Plos One QNN Light Cone
Law/Results/`; elsewhere it writes to `cohalign_outputs/` next to the
notebook. Every phase caches its output CSV and is skipped on re-run unless
`FORCE_RERUN = True`.

## Definitions used throughout (self-contained)

Architecture: the U(1)-equivariant brickwork ansatz on the cycle $C_n$; layer
$\ell$ applies trainable $R_z(\theta_{\ell q})$ on every site, then fixed XY
hopping $\exp[-i\beta_{\ell j}(X_jX_{j+1}+Y_jY_{j+1})]$ on alternating edges;
a single-qubit (or structured) Markovian channel $\Phi_\gamma$ acts once after
each layer. Readout $O_B = P_r Z_0 P_r$ in charge sector $r$; input is the
localised sector-$r$ basis state. The hopping background $\beta$ is drawn once
per architecture (`teacher_background`) and frozen; trainable phases are drawn
from the small box $\theta \sim \mathcal U[-\delta_{\mathrm{init}},
\delta_{\mathrm{init}}]^{L\times n}$.

**Restricted generator.** With $\Phi_\gamma = \mathrm{id} + \gamma
\mathcal L + O(\gamma^2)$, the study restricts $\mathcal L$ to the in-sector
off-diagonal block: $\mathcal L_r = \Pi_{\mathrm{off},r}\,\mathcal
L\,\Pi_{\mathrm{off},r}$, built numerically on the pair basis
$\{|a\rangle\langle b| : a \ne b \in \mathcal B_r\}$ as $(S - I)/
\gamma_{\mathrm{probe}}$.

**Worst-case sector coherence rate.**
$\lambda_{\mathrm{coh}} = \sup_{A\ne 0} \dfrac{-\mathrm{Re}\langle A,
\mathcal L_r A\rangle}{\langle A, A\rangle}$ — the largest eigenvalue of the
Hermitian part of $-\mathcal L_r$. Computed spectrally for **every** channel
here, structured channels included.

**Gradient mode and the operator-level aligned rate.** For parameter
$(\ell_i, q_i)$, the Heisenberg readout at the insertion point is $O_B(\mathrm{ins})
= W^\dagger O_B W$ with $W$ collecting the same layer's XY block and all later
layers; the gradient mode is $G_i(\theta) = \Pi_{\mathrm{off},r}[Z_{q_i},
O_B(\mathrm{ins};\theta)]$, forward-conjugated to each noise slot $\ell \ge
\ell_i$. The **quadratic estimator** is the small-box ensemble average of the
Rayleigh quotient of $-\mathcal L_r$ on these slot modes:
$\lambda_{\mathrm{vis}}^{\mathrm{op}} = \mathbb E_\theta\,
\overline{\mathcal R[G_i^{(\ell)}]}$.

**Response-weighted layer-resolved rates (the sharp predictor).** The exact
first-order relative degradation of the readout derivative decomposes into one
rate per noise slot,
$$r_\ell = \frac{-\mathrm{Re}\,\mathrm{Tr}\big(O_B^{(\ell)}\,\mathcal
L(D^{(\ell)})\big)}{\partial f_0} \;(\ell \ge \ell_i), \qquad
r_\ell = \frac{-\mathrm{Re}\,\mathrm{Tr}\big(F^{(\ell)}\,\mathcal
L(\rho_\ell)\big)}{\partial f_0}\;(\ell < \ell_i),$$
with $D^{(\ell)}$ the derivative-carrying operator propagated forward from the
insertion, $F^{(\ell)}$ the adjoint response functional propagated backward,
$\rho_\ell$ the noiseless state at slot $\ell$, and $\partial f_0$ the
noiseless derivative (slot-invariant — an internal frame check). First-order
prediction and alignment ratio:
$$\tilde\Delta_{\mathrm{pred}} = 2\gamma \sum_\ell r_\ell, \qquad
\omega_{\mathrm{pred}} = \frac{\sum_\ell r_\ell}{L\,\lambda_{\mathrm{coh}}},$$
directly comparable to the empirical $\hat\omega = \tilde\Delta / (2\gamma L
\lambda_{\mathrm{coh}})$ from paired common-random-number measurements.

The quadratic estimator is state-independent (a property of channel,
architecture and parameter); the response-weighted estimator additionally
weights each slot by how much of the mode the readout actually reads through
the prepared state. Phases B–F quantify when the two coincide and when the
response weighting is essential.

## Phase guide (validation ladder)

| Phase | Content | Ladder rung |
|---|---|---|
| A | Self-test battery: CPTP, sector preservation, frame invariance, bound $\lambda_{\mathrm{vis}} \le \lambda_{\mathrm{coh}}$, analytic corr-dephase rates | — |
| B | Restricted-isotropic recovery: $\lambda_{\mathrm{vis}}^{\mathrm{op}} = \lambda_{\mathrm{coh}}$ to 4 s.f. on all four isotropic channels | 1 |
| C | Zero-alignment control **predicted from the generator alone**, parameter-resolved (protected vs exposed parameters) | 2 |
| D | Structured-family audit: computed $\lambda_{\mathrm{coh}}$ (worst & typical), $\lambda_{\mathrm{vis}}^{\mathrm{op}}$, $\omega_{\mathrm{pred}}$ for all nine channels | 3 |
| E | CRN paired degradation sweep across channels × noise strengths (fresh data) | 3 |
| F | **Central validation**: predicted $\omega$ and $\tilde\Delta$ vs measured, per channel and parameter | 3 |
| G | Predictor upgrade: pooled log–log regression, $\lambda_{\mathrm{coh}}$ vs the a-priori aligned rate | 4 |
| H | Layer-resolved structure: per-slot rate profiles and depth dependence of $\omega_{\mathrm{pred}}$ | 4 |
| I | Higher charge sector $r=2$ | 5 |
| J | Estimator cost scaling | — |

Figures 1–5, a worked audit example on a user-supplied channel, the outputs
summary and the reproducibility statement follow the phases.

In [ ]:
# ============================== Configuration ==============================
import os, sys, json, time, platform
from pathlib import Path
import numpy as np
import pandas as pd

MODE = os.environ.get("COHALIGN_MODE", "paper")      # "smoke" | "paper" | "figures"
FORCE_RERUN = False
assert MODE in ("smoke", "paper", "figures")
FIGURES_ONLY = (MODE == "figures")

# ---- output location: Google Drive when on Colab, local folder otherwise ----
GDRIVE_RESULTS = ("Colab Notebooks/TEMP/Quantum Neural Networks/"
                  "Plos One QNN Light Cone Law/Results")
try:                                   # running on Google Colab?
    from google.colab import drive     # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

ROOT = Path.cwd()
if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    OUT_DIR = Path("/content/drive/MyDrive") / GDRIVE_RESULTS
else:
    OUT_DIR = ROOT / "cohalign_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / "figures";         FIG_DIR.mkdir(exist_ok=True)
# the package materialises in the runtime (fast imports; archived to OUT_DIR
# by the summary cell so the Drive folder carries the exact sources used)
PKG_DIR = ROOT / "cohalign";           PKG_DIR.mkdir(exist_ok=True)

if MODE == "paper":
    CFG = dict(
        n=8, L=3, r=1,                     # primary configuration
        depths=(3, 4, 5, 6),               # Phase H depth sweep
        gl_grid=(0.01, 0.03, 0.05, 0.10, 0.20, 0.30),
        fd_h_grid=(2e-5, 5e-5, 1e-4, 2e-4),
        fd_signed_draws=400,
        pm_L=4, pm_channels=("dephase", "corr_dephase"),
        pm_n_params=12, pm_h_grid=(2e-5, 5e-5, 1e-4),
        gl_small=(0.01, 0.03),             # small-noise window for omega_hat
        n_theta_est=40,                    # estimator ensemble size
        n_theta_meas=30,                   # CRN draws per degradation point
        preflight_draws=8,
        bootstrap_B=1000,
        r2_n=8, r2_channels=None,          # None = full suite
        cost_ns=(4, 6, 8),
    )
else:
    CFG = dict(
        n=6, L=3, r=1,
        depths=(3, 4),
        gl_grid=(0.01, 0.03, 0.10),
        fd_h_grid=(5e-5, 2e-4),
        fd_signed_draws=12,
        pm_L=4, pm_channels=("dephase",),
        pm_n_params=4, pm_h_grid=(5e-5, 2e-4),
        gl_small=(0.01, 0.03),
        n_theta_est=6,
        n_theta_meas=8,
        preflight_draws=3,
        bootstrap_B=100,
        r2_n=6, r2_channels=("amp_damp", "dephase", "corr_dephase"),
        cost_ns=(4, 6),
    )

DELTA_INIT   = 0.05          # small-box prior half-width
GAMMA_PROBE  = 1e-3          # generator probe strength
SHIFT        = np.pi / 2     # parameter-shift value
TEACHER_SEED = 42            # hopping-background seed
PREFLIGHT_SEED, EST_SEED, MEAS_SEED, BOOT_SEED = 11, 1, 1234, 7

def cache_or_compute(path, label):
    if path.exists() and not FORCE_RERUN:
        print(f"[cache] {label}: using existing {path.name}")
        return False
    if FIGURES_ONLY:
        raise FileNotFoundError(
            f"{label}: {path.name} missing but MODE='figures'. "
            "Run smoke or paper mode first.")
    return True

t_notebook_start = time.time()
print(f"CohAlign pipeline | MODE={MODE} | n={CFG['n']} L={CFG['L']} r={CFG['r']}")
print(f"environment: {'Google Colab (Drive-mounted)' if IN_COLAB else 'local'}")
print(f"outputs -> {OUT_DIR}")

## Module 1 — `cohalign_core`

Model and channel library. The cell below holds the module source, writes it
into the `cohalign/` package directory (so the repository materialises from the
notebook itself) and imports it.

In [ ]:
COHALIGN_CORE_SRC = r'''"""
cohalign_core.py -- model and noise-channel library for CohAlign.

Standalone implementation of:
  - charge-sector bookkeeping for the U(1) symmetry on n qubits
  - the U(1)-equivariant brickwork ansatz on the cycle C_n
    (trainable Rz layer followed by fixed XY hopping on alternating edges,
     single-qubit Markovian noise applied once per layer)
  - nine reference noise channels: four restricted-isotropic
    (amplitude damping, dephasing, depolarising, X-error) and five
    structured (inhomogeneous dephasing, site-dependent amplitude damping,
    biased Pauli, coherent-dissipative mix, correlated two-site dephasing)

Every channel exposes  apply(M, gamma, n)  acting on an arbitrary
2^n x 2^n matrix, so the same code path serves density-matrix evolution
and the linear-map probes used by cohalign_rates.
"""
import numpy as np
from itertools import combinations

# ---------------------------------------------------------------- Pauli
I2 = np.eye(2, dtype=complex)
PAULI_X = np.array([[0, 1], [1, 0]], dtype=complex)
PAULI_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
PAULI_Z = np.array([[1, 0], [0, -1]], dtype=complex)

# ------------------------------------------------------ sector bookkeeping
def sector_basis(n: int, r: int) -> list:
    """Computational-basis indices of Hamming weight r (bit q = site q)."""
    out = []
    for pos in combinations(range(n), r):
        s = 0
        for p in pos:
            s |= (1 << p)
        out.append(s)
    return sorted(out)

def sector_projector(n: int, r: int) -> np.ndarray:
    P = np.zeros((2**n, 2**n), dtype=complex)
    for s in sector_basis(n, r):
        P[s, s] = 1.0
    return P

def pair_basis(n: int, r: int) -> list:
    """Ordered pairs (a, b), a != b, spanning the in-sector off-diagonal block."""
    sec = sector_basis(n, r)
    return [(a, b) for a in sec for b in sec if a != b]

# ---------------------------------------------------------- embeddings
def embed_one_qubit(op: np.ndarray, q: int, n: int) -> np.ndarray:
    """Embed a 2x2 operator on qubit q (bit q of the integer index)."""
    mats = [I2] * n
    mats[n - 1 - q] = op
    out = mats[0]
    for m in mats[1:]:
        out = np.kron(out, m)
    return out

def xy_unitary(beta: float, i: int, j: int, n: int) -> np.ndarray:
    """exp(-i beta (X_i X_j + Y_i Y_j)) as a full 2^n unitary.

    Acts as identity on aligned bit pairs and as a cos/sin block on the
    swap-coupled pair, so it preserves Hamming weight exactly.
    """
    dim = 2**n
    U = np.eye(dim, dtype=complex)
    c, s = np.cos(2 * beta), np.sin(2 * beta)
    for st in range(dim):
        if ((st >> i) & 1) == ((st >> j) & 1):
            continue
        sw = st ^ (1 << i) ^ (1 << j)
        if st < sw:
            U[st, st] = c
            U[sw, sw] = c
            U[st, sw] = -1j * s
            U[sw, st] = -1j * s
    return U

# ----------------------------------------------------------- the ansatz
class Brickwork:
    """U(1)-equivariant brickwork ansatz on the cycle C_n.

    Layer ell applies trainable Rz(theta[ell, q]) on every site, then fixed
    XY hopping exp(-i beta[ell, j] (XX + YY)) on the alternating edge set
    E_ell (even edges for even ell, odd edges for odd ell, cycle-closing
    edge included). Noise, when present, acts once after each layer.
    """
    INIT_SPREAD_SEED = 202   # fixed seed of the delocalised sector input

    def __init__(self, n: int, L: int, r: int = 1, init_state: str = "localized",
                 spread_seed: int = None):
        if n < 2:
            raise ValueError("need n >= 2")
        if init_state not in ("localized", "spread"):
            raise ValueError("init_state must be 'localized' or 'spread'")
        self.n, self.L, self.r = n, L, r
        self.init_state_mode = init_state
        self.spread_seed = int(spread_seed) if spread_seed is not None \
            else self.INIT_SPREAD_SEED
        self.dim = 2**n
        self.E = [[j for j in range(n) if j % 2 == ell % 2] for ell in range(L)]
        self.P_r = sector_projector(n, r)
        self.readout = self.P_r @ embed_one_qubit(PAULI_Z, 0, n) @ self.P_r

    # -- unitaries -----------------------------------------------------
    def z_layer(self, theta_ell: np.ndarray) -> np.ndarray:
        d = np.ones(self.dim, dtype=complex)
        for q in range(self.n):
            ph_m = np.exp(-1j * theta_ell[q] / 2)
            ph_p = np.exp(+1j * theta_ell[q] / 2)
            for st in range(self.dim):
                d[st] *= ph_m if ((st >> q) & 1) == 0 else ph_p
        return np.diag(d)

    def xy_layer(self, beta_ell: np.ndarray, ell: int) -> np.ndarray:
        U = np.eye(self.dim, dtype=complex)
        for j in self.E[ell]:
            U = xy_unitary(beta_ell[j], j, (j + 1) % self.n, self.n) @ U
        return U

    def layer_unitary(self, theta: np.ndarray, beta: np.ndarray, ell: int) -> np.ndarray:
        return self.xy_layer(beta[ell], ell) @ self.z_layer(theta[ell])

    # -- states --------------------------------------------------------
    def initial_state(self) -> np.ndarray:
        """Sector-r input state.

        "localized": the basis state |1^r 0^{n-r}> (excitations on sites
        0..r-1).  NOTE: at r = 2 with the primary shallow geometry (L = 3)
        this input carries no trainable-phase response for the Z_0 readout
        to numerical precision (the activity preflight returns a map at the
        numerical floor).  The inactivity is geometry-specific -- residual
        activity of order 1e-4 reappears at greater depth -- so this is a
        guarded configuration, not a universal r >= 2 statement.
        "spread": a fixed, seeded delocalised superposition over the full
        sector basis (seed INIT_SPREAD_SEED), which restores generic
        theta-response; higher-sector validation uses this input to avoid
        the inactive configuration.
        """
        psi = np.zeros(self.dim, dtype=complex)
        if self.init_state_mode == "localized":
            psi[(1 << self.r) - 1] = 1.0
            return psi
        sec = sector_basis(self.n, self.r)
        rng = np.random.default_rng(self.spread_seed)
        amps = rng.normal(size=len(sec)) + 1j * rng.normal(size=len(sec))
        amps = amps / np.linalg.norm(amps)
        for a, s in zip(amps, sec):
            psi[s] = a
        return psi

    def evolve_dm(self, theta: np.ndarray, beta: np.ndarray,
                  channel=None, gamma: float = 0.0) -> np.ndarray:
        """rho after L layers with per-layer noise (noiseless when gamma=0)."""
        psi = self.initial_state()
        rho = np.outer(psi, psi.conj())
        for ell in range(self.L):
            U = self.layer_unitary(theta, beta, ell)
            rho = U @ rho @ U.conj().T
            if channel is not None and gamma > 0:
                rho = channel.apply(rho, gamma, self.n)
        return rho

    def output(self, rho: np.ndarray) -> float:
        return float(np.real(np.trace(self.readout @ rho)))

def teacher_background(seed: int, L: int, n: int, scale: float = 0.5) -> np.ndarray:
    """Fixed random hopping background beta (part of the architecture)."""
    rng = np.random.default_rng(seed)
    return rng.uniform(-scale, scale, size=(L, n))

# ------------------------------------------------------------- channels
def apply_single_qubit_kraus(M: np.ndarray, Ks: list, q: int, n: int) -> np.ndarray:
    """Public helper. Applies a single-qubit Kraus map on site q of an
    n-qubit operator M and returns the result."""
    return _apply_1q_kraus(M, Ks, q, n)

def _apply_1q_kraus(M: np.ndarray, Ks: list, q: int, n: int) -> np.ndarray:
    """sum_K K_q M K_q^dag on an arbitrary 2^n x 2^n matrix via tensor reshape."""
    t = M.reshape((2,) * (2 * n))
    aL, aR = n - 1 - q, 2 * n - 1 - q
    out = np.zeros_like(t)
    for K in Ks:
        tmp = np.tensordot(K, t, axes=([1], [aL]))
        tmp = np.moveaxis(tmp, 0, aL)
        tmp = np.tensordot(K.conj(), tmp, axes=([1], [aR]))
        tmp = np.moveaxis(tmp, 0, aR)
        out = out + tmp
    return out.reshape(2**n, 2**n)

class NoiseChannel:
    """Base class. Subclasses either provide kraus_single(gamma) for uniform
    per-qubit action, or override apply() for structured action."""
    name = "base"
    def kraus_single(self, gamma: float) -> list:
        raise NotImplementedError
    def apply(self, M: np.ndarray, gamma: float, n: int) -> np.ndarray:
        Ks = self.kraus_single(gamma)
        for q in range(n):
            M = _apply_1q_kraus(M, Ks, q, n)
        return M

class AmplitudeDamping(NoiseChannel):
    name = "amp_damp"
    def kraus_single(self, g):
        return [np.array([[1, 0], [0, np.sqrt(1 - g)]], dtype=complex),
                np.array([[0, np.sqrt(g)], [0, 0]], dtype=complex)]

class Dephasing(NoiseChannel):
    name = "dephase"
    def kraus_single(self, g):
        return [np.sqrt(1 - g) * I2, np.sqrt(g) * PAULI_Z]

class Depolarising(NoiseChannel):
    name = "depol"
    def kraus_single(self, g):
        return [np.sqrt(1 - g) * I2, np.sqrt(g / 3) * PAULI_X,
                np.sqrt(g / 3) * PAULI_Y, np.sqrt(g / 3) * PAULI_Z]

class XError(NoiseChannel):
    """Bit-flip channel; breaks U(1). Retained as the symmetry-breaking probe."""
    name = "x_err"
    def kraus_single(self, g):
        return [np.sqrt(1 - g) * I2, np.sqrt(g) * PAULI_X]

class InhomogeneousDephasing(NoiseChannel):
    """Per-qubit dephasing rates gamma * w_q, profile normalised to mean 1."""
    name = "inhom_dephase"
    def __init__(self, weights):
        w = np.asarray(weights, dtype=float)
        self.weights = w / np.mean(w)
    def apply(self, M, gamma, n):
        for q in range(n):
            gq = float(gamma * self.weights[q])
            if gq <= 0:
                continue
            Ks = [np.sqrt(1 - gq) * I2, np.sqrt(gq) * PAULI_Z]
            M = _apply_1q_kraus(M, Ks, q, n)
        return M

class SiteDependentAmpDamp(NoiseChannel):
    """Per-qubit amplitude-damping rates gamma * w_q, mean-normalised."""
    name = "site_amp_damp"
    def __init__(self, weights):
        w = np.asarray(weights, dtype=float)
        self.weights = w / np.mean(w)
    def apply(self, M, gamma, n):
        for q in range(n):
            gq = float(gamma * self.weights[q])
            if gq <= 0:
                continue
            Ks = [np.array([[1, 0], [0, np.sqrt(1 - gq)]], dtype=complex),
                  np.array([[0, np.sqrt(gq)], [0, 0]], dtype=complex)]
            M = _apply_1q_kraus(M, Ks, q, n)
        return M

class BiasedPauli(NoiseChannel):
    """Pauli noise with error ratios (r_X, r_Y, r_Z) summing to one."""
    # Convention. The probability tuple is ordered (p_X, p_Y, p_Z) and the
    # shipped benchmark uses (0.6, 0.2, 0.2), an X-dominated channel with
    # p_X != p_Y, which deliberately breaks phase covariance.
    name = "biased_pauli"
    def __init__(self, ratios=(0.6, 0.2, 0.2)):
        s = float(sum(ratios))
        self.ratios = tuple(x / s for x in ratios)
    def kraus_single(self, g):
        rX, rY, rZ = self.ratios
        return [np.sqrt(max(1 - g, 0.0)) * I2, np.sqrt(g * rX) * PAULI_X,
                np.sqrt(g * rY) * PAULI_Y, np.sqrt(g * rZ) * PAULI_Z]

class CoherentDissipativeMix(NoiseChannel):
    """Null coherent-component control. A uniform global Rz over-rotation
    (eps = eps_ratio * gamma, applied before amplitude damping at gamma)
    acts as a global phase within any fixed charge sector, so the coherent
    component is operationally invisible in-sector by construction and the
    channel must audit identically to amplitude damping. A genuinely active
    coherent perturbation is provided by SiteZOverRotation below."""
    name = "coh_diss_mix"
    def __init__(self, epsilon_ratio=0.5):
        self.epsilon_ratio = float(epsilon_ratio)
    def apply(self, M, gamma, n):
        eps = self.epsilon_ratio * gamma
        Urot = np.array([[np.exp(-1j * eps / 2), 0],
                         [0, np.exp(+1j * eps / 2)]], dtype=complex)
        for q in range(n):
            M = _apply_1q_kraus(M, [Urot], q, n)
        Ks = [np.array([[1, 0], [0, np.sqrt(1 - gamma)]], dtype=complex),
              np.array([[0, np.sqrt(gamma)], [0, 0]], dtype=complex)]
        for q in range(n):
            M = _apply_1q_kraus(M, Ks, q, n)
        return M

class CorrelatedDephasing(NoiseChannel):
    """Two-site ZZ dephasing on a fixed edge set:
        M -> (1 - gamma) M + gamma (Z_i Z_j) M (Z_i Z_j)   per edge.

    On the disjoint even-edge pairing {(2k, 2k+1)} in the single-excitation
    sector, coherences between sites of the SAME edge are exactly protected
    while cross-edge coherences decay -- the alignment-structured control.
    """
    name = "corr_dephase"
    def __init__(self, n, edges=None):
        if edges is None:
            edges = [(2 * k, 2 * k + 1) for k in range(n // 2)]
        self.edges = list(edges)
    def apply(self, M, gamma, n):
        for (i, j) in self.edges:
            t = M.reshape((2,) * (2 * n))
            tmp = t
            for q in (i, j):
                aL, aR = n - 1 - q, 2 * n - 1 - q
                tmp = np.tensordot(PAULI_Z, tmp, axes=([1], [aL]))
                tmp = np.moveaxis(tmp, 0, aL)
                tmp = np.tensordot(PAULI_Z.conj(), tmp, axes=([1], [aR]))
                tmp = np.moveaxis(tmp, 0, aR)
            M = ((1 - gamma) * t + gamma * tmp).reshape(2**n, 2**n)
        return M

class SiteZOverRotation(NoiseChannel):
    """Pure coherent site-dependent Z over-rotation (no dissipation).

    Applies exp(-i * gamma * c_q * Z_q / 2) on every site with the fixed
    non-uniform profile c_q = q / (n - 1). Because the profile is not
    proportional to the conserved total charge, the rotation acts
    non-trivially on in-sector coherences. Its restricted generator is
    anti-Hermitian in the Frobenius geometry, so lambda_coh vanishes and
    the audit reports a *signed* response susceptibility without a
    normalised alignment interpretation."""
    name = "site_z_overrotation"
    def apply(self, M, gamma, n):
        for q in range(n):
            c = q / (n - 1) if n > 1 else 0.0
            eps = gamma * c
            Urot = np.array([[np.exp(-1j * eps / 2), 0],
                             [0, np.exp(+1j * eps / 2)]], dtype=complex)
            M = _apply_1q_kraus(M, [Urot], q, n)
        return M

def build_channel_suite(n: int) -> dict:
    """The nine reference channels at system size n, isotropic first."""
    w = np.linspace(0.5, 1.5, n)
    return {
        "amp_damp":      AmplitudeDamping(),
        "dephase":       Dephasing(),
        "depol":         Depolarising(),
        "x_err":         XError(),
        "inhom_dephase": InhomogeneousDephasing(w),
        "site_amp_damp": SiteDependentAmpDamp(w),
        "biased_pauli":  BiasedPauli((0.6, 0.2, 0.2)),
        "coh_diss_mix":  CoherentDissipativeMix(0.5),
        "corr_dephase":  CorrelatedDephasing(n),
    }

ISOTROPIC = ("amp_damp", "dephase", "depol", "x_err")
STRUCTURED = ("inhom_dephase", "site_amp_damp", "biased_pauli",
              "coh_diss_mix", "corr_dephase")

def check_trace_preserving(channel, n: int, gamma: float = 0.05, seed: int = 0,
                           tol: float = 1e-10) -> float:
    """Max |Tr Phi(rho) - 1| over a few random density matrices."""
    rng = np.random.default_rng(seed)
    worst = 0.0
    for _ in range(3):
        A = rng.normal(size=(2**n, 2**n)) + 1j * rng.normal(size=(2**n, 2**n))
        rho = A @ A.conj().T
        rho = rho / np.trace(rho)
        worst = max(worst, abs(float(np.real(np.trace(channel.apply(rho, gamma, n)))) - 1.0))
    return worst
'''
(PKG_DIR / "cohalign_core.py").write_text(COHALIGN_CORE_SRC)
if str(PKG_DIR) not in sys.path:
    sys.path.insert(0, str(PKG_DIR))
import importlib
import cohalign_core as cac
importlib.reload(cac)
print(f"cohalign_core written and imported ({len(COHALIGN_CORE_SRC.splitlines())} lines)")

## Module 2 — `cohalign_rates`

The contribution of this study: the restricted generator, the worst-case and
typical sector coherence rates, the quadratic operator-level estimator
$\lambda_{\mathrm{vis}}^{\mathrm{op}}$, the response-weighted layer-resolved
rates $r_\ell$, and the one-call `audit_channel` API.

In [ ]:
COHALIGN_RATES_SRC = r'''"""
cohalign_rates.py -- generator-level coherence-rate estimators (the CohAlign
contribution).

Implements three objects on the in-sector off-diagonal block:

1.  lambda_coh  (worst-case sector coherence rate)
        The largest eigenvalue of the Hermitian part of -L_r, where
        L_r = (S - I)/gamma_probe and S is the channel superoperator
        restricted to the off-diagonal pair basis.  This is the
        variational worst case  sup_A -Re<A, L_r A> / <A, A>  and is
        computed here for EVERY channel, structured channels included
        (no analytic labels).

2.  lambda_vis_op  (operator-level aligned rate; quadratic estimator)
        The ensemble-averaged Rayleigh quotient of -L_r on the gradient
        mode  G_i(theta) = Pi_off,r [Z_qi, O_B(ins; theta)], where
        O_B(ins) is the readout Heisenberg-evolved back to the Rz
        insertion point (through all later layers AND the same layer's
        XY block), and the slot-ell mode is the forward conjugation of
        G_i to noise slot ell.  State-independent; this is the literal
        computable form of the aligned-rate Rayleigh quotient.

3.  response_rates  (response-weighted, layer-resolved aligned rates;
        the sharp a-priori predictor)
        Exact first-order degradation rates of the readout derivative,
        one per noise slot:
          post-insertion slots:  r_ell = -Re Tr(O_B^(ell) L_ch(D^(ell))) / df0
          pre-insertion  slots:  r_ell = -Re Tr(F^(ell)  L_ch(rho_ell)) / df0
        where D^(ell) is the derivative-carrying operator propagated
        forward from the insertion, F^(ell) is the adjoint response
        functional propagated backward, rho_ell is the noiseless state
        at slot ell, and df0 = Tr(O_B^(ell) D^(ell)) is the noiseless
        derivative (slot-invariant; used as an internal frame check).
        First-order prediction:  Delta_tilde ~= 2 * gamma * sum_ell r_ell.

The predicted alignment ratio reported by the audit is
    omega_pred = sum_ell r_ell / (L * lambda_coh),
directly comparable to the empirical  omega_hat = Delta_tilde / (2 gamma L
lambda_coh)  extracted from paired degradation measurements.
"""
import time
import numpy as np
from cohalign_core import (pair_basis, embed_one_qubit, PAULI_Z)

# ------------------------------------------------- restricted superoperator
def restricted_offdiag_superoperator(channel, n: int, r: int,
                                     gamma_probe: float = 1e-3) -> tuple:
    """Full matrix S of the channel on the in-sector off-diagonal pair basis.

    S[(c,d),(a,b)] = (Phi_gamma(|a><b|))[c,d]; off-diagonal-in-pair-basis
    leakage is captured exactly.  Returns (S, pairs).
    """
    pairs = pair_basis(n, r)
    K = len(pairs)
    dim = 2**n
    S = np.zeros((K, K), dtype=complex)
    for col, (a, b) in enumerate(pairs):
        E = np.zeros((dim, dim), dtype=complex)
        E[a, b] = 1.0
        Eo = channel.apply(E, gamma_probe, n)
        for row, (c, d) in enumerate(pairs):
            S[row, col] = Eo[c, d]
    return S, pairs

def restricted_generator(channel, n: int, r: int,
                         gamma_probe: float = 1e-3) -> tuple:
    """L_r = (S - I)/gamma_probe on the pair basis. Returns (L_r, pairs)."""
    S, pairs = restricted_offdiag_superoperator(channel, n, r, gamma_probe)
    return (S - np.eye(len(pairs))) / gamma_probe, pairs

def lambda_coh_worst(L_r: np.ndarray) -> float:
    """sup_{A != 0} -Re<A, L_r A>/<A, A> = max eig of the Hermitian part of -L_r."""
    H = -0.5 * (L_r + L_r.conj().T)
    return float(np.linalg.eigvalsh(H)[-1])

def lambda_coh_typical(L_r: np.ndarray) -> float:
    """Mean of the Hermitian-part spectrum (average contraction rate)."""
    H = -0.5 * (L_r + L_r.conj().T)
    return float(np.mean(np.linalg.eigvalsh(H)))

def rayleigh_rate(L_r: np.ndarray, g: np.ndarray) -> float:
    """-Re<g, L_r g> / <g, g> for a pair-basis vector g."""
    den = float(np.real(np.vdot(g, g)))
    if den < 1e-28:
        return float("nan")
    return float(-np.real(np.vdot(g, L_r @ g)) / den)

def vec_offdiag(M: np.ndarray, pairs: list) -> np.ndarray:
    return np.array([M[a, b] for (a, b) in pairs], dtype=complex)

# --------------------------------------------------- gradient-mode geometry
def insertion_readout(model, theta: np.ndarray, beta: np.ndarray,
                      ell_i: int) -> np.ndarray:
    """Heisenberg readout at the Rz insertion point of layer ell_i:
    evolved back through all layers > ell_i and the XY block of layer ell_i."""
    U_back = model.xy_layer(beta[ell_i], ell_i)
    for ell in range(ell_i + 1, model.L):
        U_back = model.layer_unitary(theta, beta, ell) @ U_back
    return U_back.conj().T @ model.readout @ U_back

def slot_modes(model, theta: np.ndarray, beta: np.ndarray,
               ell_i: int, q_i: int) -> dict:
    """Gradient mode G = [Z_qi, O_B(ins)] forward-conjugated to each noise
    slot ell in [ell_i, L-1]. Returns {ell: mode matrix}."""
    Zq = embed_one_qubit(PAULI_Z, q_i, model.n)
    OB_ins = insertion_readout(model, theta, beta, ell_i)
    G = Zq @ OB_ins - OB_ins @ Zq
    W = model.xy_layer(beta[ell_i], ell_i)
    M = W @ G @ W.conj().T
    modes = {ell_i: M}
    for ell in range(ell_i + 1, model.L):
        Ul = model.layer_unitary(theta, beta, ell)
        M = Ul @ M @ Ul.conj().T
        modes[ell] = M
    return modes

# ------------------------------------- estimator 1: quadratic (Eq.-7 literal)
def lambda_mode(model, beta: np.ndarray, channel, ell_i: int, q_i: int,
                  n_theta: int = 20, delta_init: float = 0.05,
                  seed: int = 1, gamma_probe: float = 1e-3,
                  L_r=None, pairs=None, theta_draws=None) -> dict:
    """Operator mode contraction diagnostic. Ensemble-averaged Rayleigh
    quotient of -L_r on the slot modes of the derivative carrying operator.
    This is a state-independent operator diagnostic, not a visibility
    measure, and it does not by itself predict the measured response.

    Returns {"lambda_vis_op", "per_slot", "lambda_coh"}; per_slot maps each
    noise slot ell >= ell_i to its mean quadratic rate.
    """
    if L_r is None or pairs is None:
        L_r, pairs = restricted_generator(channel, model.n, model.r,
                                          gamma_probe)
    lam_coh = lambda_coh_worst(L_r)
    rng = np.random.default_rng(seed)
    per = {ell: [] for ell in range(ell_i, model.L)}
    if theta_draws is not None:
        n_theta = len(theta_draws)
    for j in range(n_theta):
        theta = (theta_draws[j] if theta_draws is not None
                 else rng.uniform(-delta_init, delta_init,
                                  size=(model.L, model.n)))
        for ell, M in slot_modes(model, theta, beta, ell_i, q_i).items():
            g = vec_offdiag(M, pairs)
            v = rayleigh_rate(L_r, g)
            if np.isfinite(v):
                per[ell].append(v)
    per_slot = {ell: (float(np.mean(v)) if v else float("nan"))
                for ell, v in per.items()}
    vals = [v for v in per_slot.values() if np.isfinite(v)]
    return {"lambda_vis_op": float(np.mean(vals)) if vals else float("nan"),
            "per_slot": per_slot, "lambda_coh": lam_coh}

# ------------------- estimator 2: response-weighted, layer-resolved (sharp)
def lambda_vis_op(*args, **kwargs):
    """Deprecated alias for lambda_mode, retained for backward
    compatibility with earlier releases."""
    import warnings
    warnings.warn("lambda_vis_op is deprecated; use lambda_mode",
                  DeprecationWarning, stacklevel=2)
    return lambda_mode(*args, **kwargs)

def response_rates(model, theta: np.ndarray, beta: np.ndarray, channel,
                   ell_i: int, q_i: int, gamma_probe: float = 1e-3,
                   frame_check: bool = False) -> dict:
    """Exact first-order aligned rates, one per noise slot (all L slots).

    Returns {"rates": {ell: r_ell}, "df0": noiseless derivative,
             "frame_defect": worst frame-invariance violation if checked}.
    Delta_tilde_pred = 2 * gamma * sum(rates.values()).
    """
    n, L = model.n, model.L
    Zq = embed_one_qubit(PAULI_Z, q_i, n)
    # noiseless states after each layer; state at the insertion point
    psi = model.initial_state()
    rho = np.outer(psi, psi.conj())
    rho_slot = []
    rho_pre_ins = None
    for ell in range(L):
        Uz = model.z_layer(theta[ell])
        Ux = model.xy_layer(beta[ell], ell)
        rho_z = Uz @ rho @ Uz.conj().T
        if ell == ell_i:
            rho_pre_ins = rho_z
        rho = Ux @ rho_z @ Ux.conj().T
        rho_slot.append(rho)
    # derivative-carrying operator, forward from the insertion
    C = -0.5j * (Zq @ rho_pre_ins - rho_pre_ins @ Zq)
    Ux_i = model.xy_layer(beta[ell_i], ell_i)
    D = Ux_i @ C @ Ux_i.conj().T
    D_slot = {ell_i: D}
    for ell in range(ell_i + 1, L):
        Ul = model.layer_unitary(theta, beta, ell)
        D = Ul @ D @ Ul.conj().T
        D_slot[ell] = D
    # Heisenberg readout at each slot
    OB_slot = {L - 1: model.readout}
    for ell in range(L - 2, -1, -1):
        Ul = model.layer_unitary(theta, beta, ell + 1)
        OB_slot[ell] = Ul.conj().T @ OB_slot[ell + 1] @ Ul
    df0 = float(np.real(np.trace(OB_slot[ell_i] @ D_slot[ell_i])))
    # adjoint response functional for pre-insertion slots
    OB_ins = Ux_i.conj().T @ OB_slot[ell_i] @ Ux_i
    F = -0.5j * (OB_ins @ Zq - Zq @ OB_ins)
    Uz_i = model.z_layer(theta[ell_i])
    F = Uz_i.conj().T @ F @ Uz_i
    F_slot = {}
    if ell_i >= 1:
        F_slot[ell_i - 1] = F
        for ell in range(ell_i - 2, -1, -1):
            Ul = model.layer_unitary(theta, beta, ell + 1)
            F_slot[ell] = Ul.conj().T @ F_slot[ell + 1] @ Ul
    frame_defect = 0.0
    if frame_check and abs(df0) > 1e-16:
        for ell in range(ell_i, L):
            v = float(np.real(np.trace(OB_slot[ell] @ D_slot[ell])))
            frame_defect = max(frame_defect, abs(v - df0) / abs(df0))
        for ell in F_slot:
            v = float(np.real(np.trace(F_slot[ell] @ rho_slot[ell])))
            frame_defect = max(frame_defect, abs(v - df0) / abs(df0))
    Lch = lambda X: (channel.apply(X, gamma_probe, n) - X) / gamma_probe
    # raw first-order response terms h_ell (no division by the derivative)
    h_slots = {}
    for ell in range(ell_i, L):
        h_slots[ell] = float(np.real(np.trace(OB_slot[ell] @ Lch(D_slot[ell]))))
    for ell in F_slot:
        h_slots[ell] = float(np.real(np.trace(F_slot[ell] @ Lch(rho_slot[ell]))))
    # per-draw ratios r_ell = -h_ell / g0 as an OPTIONAL diagnostic only
    if abs(df0) > 1e-14:
        rates = {ell: float(-h / df0) for ell, h in h_slots.items()}
    else:
        rates = {ell: float("nan") for ell in h_slots}
    return {"rates": rates, "h_slots": h_slots, "df0": df0,
            "frame_defect": frame_defect}

def response_terms(model, theta, beta, channel, ell_i, q_i,
                   gamma_probe=1e-3, frame_check=False):
    """Raw first-order response terms for one draw. Returns
    {"g0": noiseless derivative, "h_slots": {ell: h_ell}, "frame_defect"}.
    The cross-moment estimator aggregates these without ever dividing by
    an individual draw."""
    out = response_rates(model, theta, beta, channel, ell_i, q_i,
                         gamma_probe, frame_check)
    return {"g0": out["df0"], "h_slots": out["h_slots"],
            "frame_defect": out["frame_defect"]}

def lambda_vis_resp(model, beta: np.ndarray, channel, ell_i: int, q_i: int,
                    n_theta: int = 20, delta_init: float = 0.05,
                    seed: int = 1, gamma_probe: float = 1e-3,
                    bootstrap_B: int = 200, bootstrap_seed=None,
                    theta_draws=None) -> dict:
    """Ensemble response-weighted rates over the small-box prior.

    Aggregation is the derivative-squared-weighted mean,
        rate_ell = sum_draws df0^2 r_ell / sum_draws df0^2,
    which is the exact first-order object matched by the paired M2 ratio:
        M2(gamma)/M2(0) = <df0^2 (1 - 2 gamma sum_ell r_ell)> / <df0^2>
                        = 1 - 2 gamma sum_ell rate_ell + O(gamma^2).
    A plain mean of per-draw rate ratios is a mean-of-ratios and becomes
    unstable whenever df0 varies strongly across the prior (deep circuits);
    the weighted form is identical where df0 is stable and remains finite
    everywhere.  The effective sample size ESS = (sum w)^2 / sum w^2 and a
    weighted-bootstrap 95% CI on rate_sum quantify ensemble uncertainty.

    Returns {"rate_sum", "rate_sum_ci": (lo, hi), "per_slot", "n_used",
             "ess"}.
    """
    if theta_draws is not None:
        n_theta = len(theta_draws)
    rng = np.random.default_rng(seed)
    # Direct cross-moment accumulation in a SINGLE pass over the draws.
    # If theta_draws is supplied it is used verbatim, which lets a matched
    # finite-difference reference share EXACTLY the same parameter draws.
    # For each slot, N_ell = sum_j (-g0_j * h_ell_j) and D = sum_j g0_j^2,
    # with Lambda_ell = N_ell / D. No per-draw division is performed, every
    # sampled draw enters, and per-draw numerator and denominator arrays are
    # retained so the bootstrap is a direct ratio-of-sums resample with no
    # second execution of the response calculation.
    Lp = model.L
    num_draw = np.zeros((n_theta, Lp))
    den_draw = np.zeros(n_theta)
    k = 0
    for j in range(n_theta):
        theta = (theta_draws[j] if theta_draws is not None
                 else rng.uniform(-delta_init, delta_init,
                                  size=(Lp, model.n)))
        t = response_terms(model, theta, beta, channel, ell_i, q_i,
                           gamma_probe)
        g0 = t["g0"]
        if not np.isfinite(g0):
            continue
        for ell, h in t["h_slots"].items():
            num_draw[k, ell] = -g0 * h
        den_draw[k] = g0 * g0
        k += 1
    num_draw, den_draw = num_draw[:k], den_draw[:k]
    if k == 0 or den_draw.sum() <= 0.0:
        return {"rate_sum": float("nan"), "rate_sum_ci": (float("nan"),) * 2,
                "per_slot": {ell: float("nan") for ell in range(Lp)},
                "n_used": 0, "ess": 0.0,
                "bootstrap_B": int(bootstrap_B),
                "bootstrap_seed": int(bootstrap_seed
                                      if bootstrap_seed is not None
                                      else seed + 10_000),
                "num_draw": num_draw, "den_draw": den_draw}
    D = float(den_draw.sum())
    per_slot = {ell: float(num_draw[:, ell].sum() / D) for ell in range(Lp)}
    rate_sum = float(num_draw.sum() / D)
    ess = float(D ** 2 / np.sum(den_draw ** 2))  # weight concentration
    if bootstrap_seed is None:
        bootstrap_seed = seed + 10_000
    brng = np.random.default_rng(bootstrap_seed)
    tot = num_draw.sum(axis=1)
    stats = []
    for _ in range(bootstrap_B):
        idx = brng.integers(0, k, size=k)
        dd = den_draw[idx].sum()
        if dd > 0:
            stats.append(float(tot[idx].sum() / dd))
    lo, hi = (np.percentile(stats, [2.5, 97.5]) if stats
              else (float("nan"), float("nan")))
    return {"rate_sum": rate_sum, "rate_sum_ci": (float(lo), float(hi)),
            "per_slot": per_slot, "n_used": int(k), "ess": ess,
            "bootstrap_B": int(bootstrap_B),
            "bootstrap_seed": int(bootstrap_seed),
            "num_draw": num_draw, "den_draw": den_draw}

def prepare_channel_context(channel, n, r, gamma_probe=1e-3):
    """Build the restricted generator ONCE and derive everything that does
    not depend on the parameter. Returns a dict context consumed by
    audit_parameter, so auditing many parameters of one channel reuses the
    expensive construction. The normalisation status is classified from the
    relative Frobenius norms of the Hermitian and anti-Hermitian parts of
    the restricted generator rather than from an absolute rate threshold."""
    L_r, pairs = restricted_generator(channel, n, r, gamma_probe)
    H = -0.5 * (L_r + L_r.conj().T)
    A = 0.5 * (L_r - L_r.conj().T)
    nH = float(np.linalg.norm(H)); nA = float(np.linalg.norm(A))
    if nH >= 10.0 * nA or nA == 0.0:
        status = "contractive"
    elif nA >= 10.0 * nH:
        status = "coherent_dominated"
    else:
        status = "mixed"
    return {"channel": channel, "n": n, "r": r,
            "gamma_probe": gamma_probe, "L_r": L_r, "pairs": pairs,
            "lambda_coh_worst": lambda_coh_worst(L_r),
            "lambda_coh_typical": lambda_coh_typical(L_r),
            "herm_norm": nH, "antiherm_norm": nA,
            "normalisation_status": status}

def lambda_mode_from_context(ctx, model, beta, ell_i, q_i, n_theta=20,
                             delta_init=0.05, seed=1, theta_draws=None):
    """Operator mode contraction diagnostic computed from a prepared channel
    context; the restricted generator is never rebuilt here."""
    return lambda_mode(model, beta, ctx["channel"], ell_i, q_i,
                         n_theta=n_theta, delta_init=delta_init, seed=seed,
                         gamma_probe=ctx["gamma_probe"],
                         L_r=ctx["L_r"], pairs=ctx["pairs"],
                         theta_draws=theta_draws)

def audit_parameter(ctx, model, beta, ell_i, q_i, n_theta=20,
                    delta_init=0.05, seed=1, theta_draws=None,
                    bootstrap_B=200, bootstrap_seed=None):
    """Audit one parameter using a prepared channel context. The generator
    is not rebuilt. omega values are reported only when the context status
    is contractive; otherwise the signed susceptibility is the output."""
    t0 = time.time()
    channel = ctx["channel"]; gamma_probe = ctx["gamma_probe"]
    lam_w = ctx["lambda_coh_worst"]; lam_t = ctx["lambda_coh_typical"]
    op = lambda_mode_from_context(ctx, model, beta, ell_i, q_i,
                                  n_theta=n_theta, delta_init=delta_init,
                                  seed=seed, theta_draws=theta_draws)
    rp = lambda_vis_resp(model, beta, channel, ell_i, q_i,
                         n_theta=n_theta, delta_init=delta_init,
                         seed=seed, gamma_probe=gamma_probe,
                         bootstrap_B=bootstrap_B,
                         bootstrap_seed=bootstrap_seed,
                         theta_draws=theta_draws)
    contractive = ctx["normalisation_status"] == "contractive" and lam_w > 0
    omega_op = op["lambda_vis_op"] / lam_w if contractive else float("nan")
    omega_resp = (rp["rate_sum"] / (model.L * lam_w)
                  if contractive else float("nan"))
    ci = rp["rate_sum_ci"]
    omega_resp_ci = (tuple(c / (model.L * lam_w) for c in ci)
                     if contractive else (float("nan"),) * 2)
    out = {"channel": getattr(channel, "name", type(channel).__name__),
           "n": model.n, "L": model.L, "r": model.r,
           "ell_i": ell_i, "q_i": q_i,
           "lambda_coh_worst": lam_w, "lambda_coh_typical": lam_t,
           "normalisation_status": ctx["normalisation_status"],
           "lambda_mode": op["lambda_vis_op"], "omega_mode": omega_op,
           "lambda_response": rp["rate_sum"],
           "lambda_response_ci": rp["rate_sum_ci"],
           "omega_response": omega_resp, "omega_response_ci": omega_resp_ci,
           "per_slot_response": rp["per_slot"],
           "signed_susceptibility_only": not contractive,
           "ess": rp["ess"], "n_theta_used": rp["n_used"],
           "bootstrap_B": rp["bootstrap_B"],
           "bootstrap_seed": rp["bootstrap_seed"],
           "wall_seconds": time.time() - t0,
           # deprecated aliases retained for backward compatibility
           "lambda_vis_op": op["lambda_vis_op"], "omega_op": omega_op,
           "rate_sum_resp": rp["rate_sum"], "omega_resp": omega_resp,
           "omega_resp_ci": omega_resp_ci,
           "per_slot_resp": rp["per_slot"], "per_slot_op": op["per_slot"]}
    if not contractive:
        out["note"] = ("normalisation status is "
                       f"{ctx['normalisation_status']}; the audit reports "
                       "the signed response susceptibility and no "
                       "normalised alignment ratio")
    return out

def audit_parameters(ctx, model, beta, params, n_theta=20,
                     delta_init=0.05, seed=1):
    """Audit a list of (ell, q) parameters with one shared context."""
    return [audit_parameter(ctx, model, beta, e, q, n_theta=n_theta,
                            delta_init=delta_init, seed=seed)
            for (e, q) in params]

def audit_channel(model, beta: np.ndarray, channel, ell_i: int, q_i: int,
                  n_theta: int = 20, delta_init: float = 0.05,
                  seed: int = 1, gamma_probe: float = 1e-3) -> dict:
    """One-call audit of a (channel, architecture, parameter) triple.
    Equivalent to prepare_channel_context followed by audit_parameter; the
    restricted generator is built exactly once."""
    t0 = time.time()
    ctx = prepare_channel_context(channel, model.n, model.r, gamma_probe)
    out = audit_parameter(ctx, model, beta, ell_i, q_i, n_theta=n_theta,
                          delta_init=delta_init, seed=seed)
    out["wall_seconds"] = time.time() - t0
    return out


'''
(PKG_DIR / "cohalign_rates.py").write_text(COHALIGN_RATES_SRC)
import cohalign_rates as car
importlib.reload(car)
print(f"cohalign_rates written and imported ({len(COHALIGN_RATES_SRC.splitlines())} lines)")

## Module 3 — `cohalign_bench`

Empirical validation machinery: activity preflight, common-random-number
paired degradation with bootstrap, the empirical alignment ratio, and the
regression helper.

In [ ]:
COHALIGN_BENCH_SRC = r'''"""
cohalign_bench.py -- empirical validation machinery for CohAlign.

Provides:
  - activity_preflight / pick_parameters: locate readout-visible (active)
    parameters via the noiseless parameter-shift gradient map, and select
    validation parameters spanning the alignment range
  - paired_degradation: common-random-numbers paired measurement of the
    relative squared-gradient degradation Delta_tilde = 1 - M2(gamma)/M2(0)
    with bootstrap confidence intervals (shared theta draws between the
    noiseless and noisy passes, and across channels)
  - omega_hat: empirical alignment ratio Delta_tilde / (2 gamma L lambda_coh)
  - ols_loglog: ordinary least squares on log-log design matrices with R^2,
    RMSE and coefficient table (used for the predictor-comparison phase)
"""
import numpy as np

# ------------------------------------------------------ activity preflight
def parameter_shift_grad(model, theta, beta, ell, q, channel=None,
                         gamma=0.0, shift=np.pi / 2):
    tp, tm = theta.copy(), theta.copy()
    tp[ell, q] += shift
    tm[ell, q] -= shift
    fp = model.output(model.evolve_dm(tp, beta, channel, gamma))
    fm = model.output(model.evolve_dm(tm, beta, channel, gamma))
    return 0.5 * (fp - fm)

def activity_preflight(model, beta, n_draw=4, delta_init=0.05, seed=11):
    """Mean squared noiseless gradient for every (ell, q). O(n L) circuit pairs
    per draw; identifies the backward light cone of the readout."""
    act = np.zeros((model.L, model.n))
    rng = np.random.default_rng(seed)
    for _ in range(n_draw):
        theta = rng.uniform(-delta_init, delta_init, size=(model.L, model.n))
        for ell in range(model.L):
            for q in range(model.n):
                g = parameter_shift_grad(model, theta, beta, ell, q)
                act[ell, q] += g * g
    return act / n_draw

def pick_parameters(act, k=2, floor_frac=1e-4, abs_floor=1e-20,
                    distinct_layers=True):
    """Top-k active parameters by mean squared gradient (descending).

    With distinct_layers=True (default) the selected parameters are drawn
    from k different layers, so validation covers genuinely distinct (not
    symmetry-equivalent) parameter locations.

    Raises if the whole map is numerically zero -- the signature of an input
    state that carries no trainable-phase response for this readout at this
    geometry (observed for the localised basis-state input at r = 2, L = 3;
    the inactivity is geometry-specific, not universal for r >= 2)."""
    if act.max() < abs_floor:
        raise RuntimeError(
            "activity preflight found no readout-visible parameters "
            f"(max activity {act.max():.2e}); the input state carries no "
            "trainable-phase response for this readout at this geometry -- "
            "for higher sectors consider Brickwork(..., init_state='spread')")
    flat = [(-act[ell, q], ell, q)
            for ell in range(act.shape[0]) for q in range(act.shape[1])
            if act[ell, q] > floor_frac * act.max()]
    flat.sort()
    if not distinct_layers:
        return [(ell, q) for (_, ell, q) in flat[:k]]
    out, used_layers = [], set()
    for (_, ell, q) in flat:
        if ell in used_layers:
            continue
        out.append((ell, q)); used_layers.add(ell)
        if len(out) == k:
            break
    # fall back to plain top-k if fewer than k layers are active
    if len(out) < k:
        out = [(ell, q) for (_, ell, q) in flat[:k]]
    return out

# ------------------------------------------------- CRN paired degradation
def paired_degradation(model, beta, ell, q, channel, gamma,
                       n_theta=8, delta_init=0.05, seed=1234,
                       shift=np.pi / 2, bootstrap_B=200, boot_seed=7,
                       g0_floor=1e-12, return_draws=False,
                       theta_draws=None):
    """CRN paired estimate of Delta_tilde = 1 - M2(gamma)/M2(0).

    The SAME theta draws feed the noiseless and noisy passes (and, because
    the seed is caller-fixed, the same draws are shared across channels and
    gamma values).  M2 is the mean squared parameter-shift derivative over
    the retained draws; the bootstrap resamples draw indices.
    Returns dict with m2_zero, m2_noise, delta_tilde, ci_lo, ci_hi, n_used;
    with return_draws=True also g0_sq and gn_sq (per-draw squared gradients,
    aligned across gamma values by the shared CRN seed) for joint bootstraps.
    """
    rng = np.random.default_rng(seed)
    g0_sq, gn_sq = [], []
    if theta_draws is not None:
        n_theta = len(theta_draws)
    for j in range(n_theta):
        theta = (theta_draws[j] if theta_draws is not None
                 else rng.uniform(-delta_init, delta_init, size=(model.L, model.n)))
        g0 = parameter_shift_grad(model, theta, beta, ell, q, None, 0.0, shift)
        if g0 * g0 < g0_floor:
            continue
        gn = parameter_shift_grad(model, theta, beta, ell, q, channel, gamma, shift)
        g0_sq.append(g0 * g0)
        gn_sq.append(gn * gn)
    g0_sq, gn_sq = np.asarray(g0_sq), np.asarray(gn_sq)
    if len(g0_sq) == 0:
        return {"m2_zero": np.nan, "m2_noise": np.nan, "delta_tilde": np.nan,
                "ci_lo": np.nan, "ci_hi": np.nan, "n_used": 0}
    m2_0, m2_g = float(np.mean(g0_sq)), float(np.mean(gn_sq))
    delta = 1.0 - m2_g / m2_0
    brng = np.random.default_rng(boot_seed)
    stats = []
    for _ in range(bootstrap_B):
        idx = brng.integers(0, len(g0_sq), size=len(g0_sq))
        stats.append(1.0 - np.mean(gn_sq[idx]) / np.mean(g0_sq[idx]))
    lo, hi = np.percentile(stats, [2.5, 97.5])
    out = {"m2_zero": m2_0, "m2_noise": m2_g, "delta_tilde": float(delta),
           "ci_lo": float(lo), "ci_hi": float(hi), "n_used": int(len(g0_sq))}
    if return_draws:
        out["g0_sq"] = g0_sq
        out["gn_sq"] = gn_sq
    return out

def omega_hat(delta_tilde, gamma, L, lambda_coh):
    """Empirical alignment ratio Delta_tilde / (2 gamma L lambda_coh)."""
    den = 2.0 * gamma * L * lambda_coh
    return float(delta_tilde / den) if den > 0 else float("nan")

# --------------------------------------------------------------- regression
def ols_loglog(y, X_cols, names):
    """OLS of y on [1, X_cols...]. Returns dict with R2, RMSE, coefficients."""
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(y))] + [np.asarray(c, dtype=float)
                                             for c in X_cols])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    yhat = X @ b
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    return {"R2": 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan,
            "RMSE": float(np.sqrt(ss_res / len(y))),
            "coef": dict(zip(["intercept"] + list(names), [float(v) for v in b])),
            "n": int(len(y))}
'''
(PKG_DIR / "cohalign_bench.py").write_text(COHALIGN_BENCH_SRC)
import cohalign_bench as cab
importlib.reload(cab)
print(f"cohalign_bench written and imported ({len(COHALIGN_BENCH_SRC.splitlines())} lines)")

# shared architecture objects for the primary configuration
N, L, R = CFG["n"], CFG["L"], CFG["r"]
MODEL = cac.Brickwork(N, L, R)
BETA  = cac.teacher_background(TEACHER_SEED, L, N)
SUITE = cac.build_channel_suite(N)
print(f"primary architecture: n={N}, L={L}, r={R}; channels: {list(SUITE)}")

# Phase A — self-test battery

Shipped unit tests. Any failure raises immediately: (i) trace preservation of
all nine channels; (ii) charge-sector preservation of the noiseless circuit;
(iii) frame invariance of the response decomposition
($\mathrm{Tr}(O_B^{(\ell)} D^{(\ell)}) = \mathrm{Tr}(F^{(\ell)} \rho_\ell) =
\partial f_0$ at every slot); (iv) the variational bound
$\mathcal R[A] \le \lambda_{\mathrm{coh}}$ on random pair-basis directions;
(v) the analytic within-edge / cross-edge rates of the correlated-dephasing
generator (0 and $\approx 4$) that make the zero-alignment construction
possible.

In [ ]:
out_csv_A = OUT_DIR / "phaseA_selftests.csv"
if cache_or_compute(out_csv_A, "Phase A"):
    print("=" * 72); print("PHASE A -- SELF-TEST BATTERY"); print("=" * 72)
    rows = []
    t0 = time.time()

    # (i) trace preservation
    for name, ch in SUITE.items():
        d = cac.check_trace_preserving(ch, min(N, 5), gamma=0.05)
        rows.append({"test": f"TP_{name}", "value": d, "tol": 1e-10, "pass": d < 1e-10})

    # (ii) sector preservation (noiseless)
    rng = np.random.default_rng(0)
    theta = rng.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
    rho = MODEL.evolve_dm(theta, BETA)
    leak = float(np.real(np.trace(rho)) - np.real(np.trace(MODEL.P_r @ rho @ MODEL.P_r)))
    rows.append({"test": "sector_preservation", "value": abs(leak),
                 "tol": 1e-10, "pass": abs(leak) < 1e-10})

    # (iii) frame invariance of the response decomposition
    worst_frame = 0.0
    for (ell_i, q_i) in [(1, 0), (L - 1, 0)]:
        out = car.response_rates(MODEL, theta, BETA, SUITE["dephase"],
                                 ell_i, q_i, GAMMA_PROBE, frame_check=True)
        if abs(out["df0"]) > 1e-12:
            worst_frame = max(worst_frame, out["frame_defect"])
    rows.append({"test": "frame_invariance", "value": worst_frame,
                 "tol": 1e-8, "pass": worst_frame < 1e-8})

    # (iv) Rayleigh bound on random directions, every channel
    for name, ch in SUITE.items():
        L_r, pairs = car.restricted_generator(ch, N, R, GAMMA_PROBE)
        lam_w = car.lambda_coh_worst(L_r)
        rng = np.random.default_rng(1)
        excess = 0.0
        for _ in range(30):
            g = rng.normal(size=len(pairs)) + 1j * rng.normal(size=len(pairs))
            v = car.rayleigh_rate(L_r, g)
            excess = max(excess, v - lam_w)
        rows.append({"test": f"bound_{name}", "value": excess,
                     "tol": 1e-8, "pass": excess < 1e-8})

    # (v) analytic corr-dephase structure: within-edge 0, cross-edge ~ 4
    cd = SUITE["corr_dephase"]
    L_r, pairs = car.restricted_generator(cd, N, R, GAMMA_PROBE)
    sec = cac.sector_basis(N, R)
    def unit_pair(a, b):
        g = np.zeros(len(pairs), dtype=complex); g[pairs.index((a, b))] = 1.0
        return g
    r_within = car.rayleigh_rate(L_r, unit_pair(1 << 0, 1 << 1))
    r_cross  = car.rayleigh_rate(L_r, unit_pair(1 << 0, 1 << 2))
    rows.append({"test": "corr_within_edge_rate", "value": abs(r_within),
                 "tol": 1e-8, "pass": abs(r_within) < 1e-8})
    rows.append({"test": "corr_cross_edge_rate", "value": abs(r_cross - 4.0),
                 "tol": 0.05, "pass": abs(r_cross - 4.0) < 0.05})

    dfA = pd.DataFrame(rows)
    dfA.to_csv(out_csv_A, index=False)
    n_fail = int((~dfA["pass"]).sum())
    print(dfA.to_string(index=False, float_format=lambda v: f"{v:.3e}"))
    print(f"\nPhase A: {len(dfA) - n_fail}/{len(dfA)} tests passed "
          f"({time.time() - t0:.1f}s)")
    assert n_fail == 0, f"{n_fail} self-tests FAILED"
else:
    dfA = pd.read_csv(out_csv_A)
    assert (~dfA["pass"]).sum() == 0

# Phase B — restricted-isotropic recovery (ladder rung 1)

On the four restricted-isotropic channels the aligned rate must equal the
worst-case rate *identically*: the generator contracts every in-sector
off-diagonal direction at the same speed, so the Rayleigh quotient is constant
on the whole block. The estimator is required to return
$\lambda_{\mathrm{vis}}^{\mathrm{op}} = \lambda_{\mathrm{coh}}$ within a
**relative tolerance of $10^{-3}$** at the working probe strength (the
finite-difference generator carries an $O(\gamma_{\mathrm{probe}})$ bias, so
the phase also reports a probe-extrapolated limit), with the isotropy itself
certified by the spread of the Rayleigh quotient over random probe directions.
Validation parameters are drawn from **distinct layers** so the family-wide
phases cover genuinely different parameter locations.

In [ ]:
out_csv_B = OUT_DIR / "phaseB_isotropic_recovery.csv"
if cache_or_compute(out_csv_B, "Phase B"):
    print("=" * 72); print("PHASE B -- RESTRICTED-ISOTROPIC RECOVERY"); print("=" * 72)
    t0 = time.time()
    act = cab.activity_preflight(MODEL, BETA, n_draw=CFG["preflight_draws"],
                                 delta_init=DELTA_INIT, seed=PREFLIGHT_SEED)
    PARAMS = cab.pick_parameters(act, k=2, distinct_layers=True)
    (ELL_A, Q_A) = PARAMS[0]
    print(f"activity preflight: validation parameters {PARAMS} "
          f"(primary = ({ELL_A},{Q_A}); layers are distinct by construction)")
    rows, probe_rows = [], []
    for name in cac.ISOTROPIC:
        ch = SUITE[name]
        # probe-size convergence: the finite-difference generator carries an
        # O(gamma_probe) bias, so compute at three probes and Richardson-
        # extrapolate to the gamma_probe -> 0 limit
        lam_by_probe = {}
        for gp in (1e-2, 1e-3, 1e-4, 1e-5):
            L_r, pairs = car.restricted_generator(ch, N, R, gp)
            lam_by_probe[gp] = car.lambda_coh_worst(L_r)
            probe_rows.append({"channel": name, "gamma_probe": gp,
                               "lambda_coh": lam_by_probe[gp]})
        lam_w = lam_by_probe[GAMMA_PROBE]
        lam_extrap = (10 * lam_by_probe[1e-5] - lam_by_probe[1e-4]) / 9
        L_r, pairs = car.restricted_generator(ch, N, R, GAMMA_PROBE)
        # isotropy certificate from the full spectrum of the Hermitian part
        H = -0.5 * (L_r + L_r.conj().T)
        ev = np.linalg.eigvalsh(H)
        spec_aniso = float((ev[-1] - ev[0]) / max(ev[-1], 1e-30))
        rng = np.random.default_rng(2)
        qs = [car.rayleigh_rate(L_r, rng.normal(size=len(pairs))
                                + 1j * rng.normal(size=len(pairs)))
              for _ in range(30)]
        op = car.lambda_mode(MODEL, BETA, ch, ELL_A, Q_A,
                               n_theta=CFG["n_theta_est"],
                               delta_init=DELTA_INIT, seed=EST_SEED,
                               gamma_probe=GAMMA_PROBE)
        ratio = op["lambda_vis_op"] / lam_w
        rows.append({"channel": name, "lambda_coh": lam_w,
                     "lambda_coh_probe_extrap": lam_extrap,
                     "lambda_vis_op": op["lambda_vis_op"], "omega_mode": ratio,
                     "rel_recovery_err": abs(ratio - 1.0),
                     "spectral_anisotropy": spec_aniso,
                     "probe_spread": max(qs) - min(qs),
                     "recovered_tol": abs(ratio - 1.0) < 1e-3})
    dfB = pd.DataFrame(rows)
    dfB.to_csv(out_csv_B, index=False)
    pd.DataFrame(probe_rows).to_csv(OUT_DIR / "phaseB_probe_convergence.csv",
                                    index=False)
    print(dfB.to_string(index=False,
          float_format=lambda v: f"{v:.5f}"))
    assert dfB["recovered_tol"].all(), "isotropic recovery outside tolerance"
    print(f"\nPhase B: isotropic recovery within relative tolerance 1e-3 "
          f"on all four channels ({time.time() - t0:.1f}s)")
else:
    dfB = pd.read_csv(out_csv_B)
    act = cab.activity_preflight(MODEL, BETA, n_draw=CFG["preflight_draws"],
                                 delta_init=DELTA_INIT, seed=PREFLIGHT_SEED)
    PARAMS = cab.pick_parameters(act, k=2, distinct_layers=True)
    (ELL_A, Q_A) = PARAMS[0]

# Phase C — zero-alignment control predicted from the generator alone (rung 2)

The correlated-dephasing channel on the disjoint even-edge pairing is the
alignment-structured control: within-edge coherences are exactly protected
(Phase A test v) while cross-edge coherences decay, so
$\lambda_{\mathrm{coh}}$ is large while the *visible* rate can vanish. Here the estimator must return this vanishing **from the
generator and the architecture alone**, where a naive spectrum-only
estimator fails — and, because the gradient mode depends
on the parameter location, the prediction is *parameter-resolved*: last-layer
parameters (whose modes live on the layer's own even edges = the protected
pairs) are predicted protected, earlier odd-layer parameters exposed. A
single small-noise paired measurement per parameter closes the loop. The
candidate set deliberately includes odd-layer parameters: on these the
state-independent quadratic estimator can report large alignment
($\omega^{\mathrm{op}} \sim 1$) for modes whose readout-visible component is
in fact protected -- the table's $\omega^{\mathrm{op}}$ vs
$\omega^{\mathrm{resp}}$ columns exhibit exactly the misassignment a
spectrum-only diagnostic suffers, and the measurement adjudicates in favour
of the response-weighted rate.

In [ ]:
out_csv_C = OUT_DIR / "phaseC_zero_alignment.csv"
if cache_or_compute(out_csv_C, "Phase C"):
    print("=" * 72); print("PHASE C -- ZERO-ALIGNMENT CONTROL, PARAMETER-RESOLVED"); print("=" * 72)
    t0 = time.time()
    cd = SUITE["corr_dephase"]
    L_r, _ = car.restricted_generator(cd, N, R, GAMMA_PROBE)
    lam_w = car.lambda_coh_worst(L_r)
    # candidate parameters: the preflight set, the last-layer readout-cone
    # sites (protected by construction when L-1 is an even layer), and two
    # odd-layer sites -- the exposed cases on which the state-independent
    # quadratic estimator misassigns alignment and the response weighting is
    # required (the contrast that motivates estimator 2)
    cand = list(dict.fromkeys(PARAMS + [(L - 1, 0), (L - 1, 1),
                                        (1, 0), (1, 1)]))
    gl_probe = CFG["gl_small"][0]
    rows = []
    for (ell_i, q_i) in cand:
        aud = car.audit_channel(MODEL, BETA, cd, ell_i, q_i,
                                n_theta=CFG["n_theta_est"],
                                delta_init=DELTA_INIT, seed=EST_SEED,
                                gamma_probe=GAMMA_PROBE)
        meas = cab.paired_degradation(MODEL, BETA, ell_i, q_i, cd,
                                      gl_probe / L,
                                      n_theta=CFG["n_theta_meas"],
                                      delta_init=DELTA_INIT, seed=MEAS_SEED,
                                      shift=SHIFT, bootstrap_B=CFG["bootstrap_B"],
                                      boot_seed=BOOT_SEED)
        if meas["n_used"] == 0:
            continue   # parameter inactive for this input; nothing to validate
        rows.append({"ell": ell_i, "q": q_i, "lambda_coh": lam_w,
                     "omega_op": aud["omega_op"],
                     "omega_resp_pred": aud["omega_resp"],
                     "delta_pred": 2 * (gl_probe / L) * aud["rate_sum_resp"],
                     "delta_meas": meas["delta_tilde"],
                     "delta_ci_lo": meas["ci_lo"], "delta_ci_hi": meas["ci_hi"],
                     "protected_pred": abs(aud["omega_resp"]) < 1e-6})
    dfC = pd.DataFrame(rows)
    dfC.to_csv(out_csv_C, index=False)
    print(dfC.to_string(index=False, float_format=lambda v: f"{v: .5f}"))
    prot = dfC[dfC["protected_pred"]]
    if len(prot):
        worst = prot["delta_meas"].abs().max()
        print(f"\npredicted-protected parameters: {len(prot)}; "
              f"worst measured |Delta| = {worst:.2e}")
        assert worst < 5e-3, "protected parameter shows measurable degradation"
    # signed-susceptibility demonstration on a pure coherent channel
    demo_ch = cac.SiteZOverRotation()
    demo = car.audit_channel(MODEL, BETA, demo_ch, ELL_A, Q_A,
                             n_theta=CFG["n_theta_est"],
                             delta_init=DELTA_INIT, seed=EST_SEED,
                             gamma_probe=GAMMA_PROBE)
    pd.DataFrame([{"channel": demo["channel"],
                   "note": "small_ensemble_diagnostic",
                   "n_draws": CFG["n_theta_est"],
                   "lambda_coh_worst": demo["lambda_coh_worst"],
                   "lambda_response": demo["lambda_response"],
                   "signed_only": demo["signed_susceptibility_only"],
                   **{f"slot{k}": v for k, v in
                      demo["per_slot_response"].items()}}]).to_csv(
        OUT_DIR / "phaseC_signed_smallensemble_diagnostic.csv", index=False)
    print("\nsigned-susceptibility demonstration (pure coherent site-Z "
          "over-rotation):")
    print(f"  lambda_coh = {demo['lambda_coh_worst']:.2e} (numerically zero)")
    print(f"  signed Lambda_resp = {demo['lambda_response']:+.4f}")
    print(f"  audit flag: {demo.get('note', 'ratio reported (contractive generator)')}")
    print(f"Phase C done ({time.time() - t0:.1f}s)")
else:
    dfC = pd.read_csv(out_csv_C)

# Phase D — structured-family audit (rung 3, predictions)

The full nine-channel audit at the primary parameter: worst-case and typical
$\lambda_{\mathrm{coh}}$ **computed spectrally for every channel** — no rate in this project is
assigned analytically — together with the quadratic estimator,
the response-weighted rate sum, and the predicted alignment ratio
$\omega_{\mathrm{pred}}$. These are the *a priori* predictions that Phase F
tests against fresh measurements.

In [ ]:
out_csv_D = OUT_DIR / "phaseD_structured_audit.csv"
if cache_or_compute(out_csv_D, "Phase D"):
    print("=" * 72); print("PHASE D -- NINE-CHANNEL GENERATOR-LEVEL AUDIT"); print("=" * 72)
    t0 = time.time()
    rows = []
    for name, ch in SUITE.items():
        aud = car.audit_channel(MODEL, BETA, ch, ELL_A, Q_A,
                                n_theta=CFG["n_theta_est"],
                                delta_init=DELTA_INIT, seed=EST_SEED,
                                gamma_probe=GAMMA_PROBE)
        rows.append({"channel": name,
                     "family": "isotropic" if name in cac.ISOTROPIC else "structured",
                     "lambda_coh_worst": aud["lambda_coh_worst"],
                     "lambda_coh_typical": aud["lambda_coh_typical"],
                     "lambda_vis_op": aud["lambda_vis_op"],
                     "omega_op": aud["omega_op"],
                     "rate_sum_resp": aud["rate_sum_resp"],
                     "omega_resp_pred": aud["omega_resp"],
                     "audit_seconds": aud["wall_seconds"]})
        print(f"  {name:14s} lam_coh={aud['lambda_coh_worst']:7.4f} "
              f"lam_vis_op={aud['lambda_vis_op']:7.4f} "
              f"omega_pred={aud['omega_resp']:7.4f}  ({aud['wall_seconds']:.1f}s)")
    dfD = pd.DataFrame(rows)
    dfD.to_csv(out_csv_D, index=False)
    print(f"\nPhase D done ({time.time() - t0:.1f}s)")
else:
    dfD = pd.read_csv(out_csv_D)

# Phase E — CRN paired degradation sweep (rung 3, measurements)

Fresh paired measurements of $\tilde\Delta = 1 - M_2(\gamma)/M_2(0)$ for every
channel across the $\gamma L$ grid at the two validation parameters, with
common random numbers (identical $\theta$ draws across channels and noise
strengths) and bootstrap confidence intervals. These data are generated by
this notebook at its own configuration — no archived measurements enter.

In [ ]:
out_csv_E = OUT_DIR / "phaseE_degradation.csv"
if cache_or_compute(out_csv_E, "Phase E"):
    print("=" * 72); print("PHASE E -- CRN PAIRED DEGRADATION SWEEP"); print("=" * 72)
    t0 = time.time()
    rows = []
    draw_store = {}
    lam_map = dict(zip(dfD["channel"], dfD["lambda_coh_worst"]))
    for name, ch in SUITE.items():
        for (ell_i, q_i) in PARAMS:
            for gl in CFG["gl_grid"]:
                gamma = gl / L
                meas = cab.paired_degradation(
                    MODEL, BETA, ell_i, q_i, ch, gamma,
                    n_theta=CFG["n_theta_meas"], delta_init=DELTA_INIT,
                    seed=MEAS_SEED, shift=SHIFT,
                    bootstrap_B=CFG["bootstrap_B"], boot_seed=BOOT_SEED,
                    return_draws=True)
                if meas["n_used"] == 0:
                    continue
                draw_store.setdefault((name, ell_i, q_i), {})[gl] = (
                    meas.pop("g0_sq"), meas.pop("gn_sq"))
                rows.append({"channel": name, "ell": ell_i, "q": q_i,
                             "gamma_L": gl, "gamma": gamma,
                             "lambda_coh": lam_map[name], **meas})
        print(f"  {name:14s} done  ({time.time() - t0:.0f}s elapsed)")
    dfE = pd.DataFrame(rows)
    dfE["omega_hat"] = dfE.apply(
        lambda r: cab.omega_hat(r["delta_tilde"], r["gamma"], L, r["lambda_coh"]),
        axis=1)
    dfE.to_csv(out_csv_E, index=False)
    # per-draw arrays for the joint bootstrap in Phase F (CRN-aligned draws)
    npz = {}
    for (name, ell_i, q_i), by_gl in draw_store.items():
        for gl, (g0, gn) in by_gl.items():
            key = f"{name}|{ell_i}|{q_i}|{gl}"
            npz[key + "|g0"] = g0
            npz[key + "|gn"] = gn
    np.savez_compressed(OUT_DIR / "phaseE_draws.npz", **npz)
    print(f"\nPhase E: {len(dfE)} measurement rows + per-draw archive "
          f"({time.time() - t0:.1f}s)")
else:
    dfE = pd.read_csv(out_csv_E)

# Phase F — central validation: predicted vs measured alignment (rung 3)

The closed loop. For every (channel, parameter) pair the *a priori* audit of
Phases C–D supplies $\omega_{\mathrm{pred}}$ and
$\tilde\Delta_{\mathrm{pred}}(\gamma) = 2\gamma\sum_\ell r_\ell$; Phase E
supplies the measured $\hat\omega$ and $\tilde\Delta_{\mathrm{meas}}(\gamma)$;
Because any finite-noise ratio carries an $O(\gamma)$ second-order bias, the
headline $\hat\omega$ is the $\gamma \to 0$ intercept of a per-channel linear
fit over the **accumulated-defect window** $\gamma L \lambda_{\mathrm{coh}}
\leq 0.4$ (prespecified: the defect, not $\gamma L$ alone, controls the
expansion, so fast and slow channels are windowed equivalently), with
tightening checks at $0.8$ and $0.2$. A joint bootstrap over the shared CRN
draws (resampling the same draw indices across every noise strength) supplies
a confidence interval for each measured intercept. Parameters selected from
distinct layers make the channel cases genuinely distinct; identical cases
are deduplicated before summary statistics. Agreement on the diagonal — including
the predicted-protected points at the origin — is the study's central result
(Figure 2).

In [ ]:
out_csv_F = OUT_DIR / "phaseF_validation.csv"
if cache_or_compute(out_csv_F, "Phase F"):
    print("=" * 72); print("PHASE F -- PREDICTED VS MEASURED ALIGNMENT"); print("=" * 72)
    t0 = time.time()
    DEFECT_PRIMARY = 0.4            # prespecified accumulated-defect cutoff
    DEFECT_CHECKS = (0.8, 0.2)      # tightening checks
    draws = np.load(OUT_DIR / "phaseE_draws.npz")
    GL_ALL = sorted(set(dfE["gamma_L"]))

    def intercept_from_draws(name, ell_i, q_i, lam, idx=None, c_def=0.4):
        gls, oms = [], []
        for gl in GL_ALL:
            key = f"{name}|{ell_i}|{q_i}|{gl}"
            if key + "|g0" not in draws.files or gl * lam > c_def:
                continue
            g0 = draws[key + "|g0"]; gn = draws[key + "|gn"]
            if idx is not None:
                g0, gn = g0[idx], gn[idx]
            m0 = np.mean(g0)
            if m0 <= 0:
                continue
            delta = 1.0 - np.mean(gn) / m0
            gls.append(gl); oms.append(delta / (2 * gl * lam))
        if len(gls) >= 3:
            return float(np.polyfit(gls, oms, 1)[1]), len(gls)
        if gls:
            return float(oms[0]), len(gls)
        return float("nan"), 0

    rows = []
    boot_store = {}
    # primary-parameter analysis; the secondary location is retained only as
    # an equivalence check reported alongside (SI), not pooled
    equiv_rows = []
    for name, ch in SUITE.items():
        for (ell_i, q_i) in PARAMS[:1]:
            aud = car.audit_channel(MODEL, BETA, ch, ell_i, q_i,
                                    n_theta=CFG["n_theta_est"],
                                    delta_init=DELTA_INIT, seed=EST_SEED,
                                    gamma_probe=GAMMA_PROBE)
            sub = dfE[(dfE["channel"] == name) & (dfE["ell"] == ell_i)
                      & (dfE["q"] == q_i)]
            if not len(sub):
                continue
            lam = float(aud["lambda_coh_worst"])
            om0, npts = intercept_from_draws(name, ell_i, q_i, lam,
                                             c_def=DEFECT_PRIMARY)
            n_used = int(sub["n_used"].iloc[0])
            brng = np.random.default_rng(BOOT_SEED + 1)
            stats = []
            for _ in range(min(CFG["bootstrap_B"], 500)):
                idx = brng.integers(0, n_used, size=n_used)
                v, _ = intercept_from_draws(name, ell_i, q_i, lam, idx=idx,
                                            c_def=DEFECT_PRIMARY)
                if np.isfinite(v):
                    stats.append(v)
            om0_lo, om0_hi = (np.percentile(stats, [2.5, 97.5])
                              if stats else (np.nan, np.nan))
            if stats and (ell_i, q_i) == PARAMS[0]:
                boot_store[name] = stats
            om_checks = {}
            for c in DEFECT_CHECKS:
                v_c, n_c = intercept_from_draws(name, ell_i, q_i, lam,
                                                c_def=c)
                # per-channel intercepts are reported only for genuine fits
                om_checks[c] = v_c if n_c >= 3 else float("nan")
            small = sub[sub["gamma_L"].isin(CFG["gl_small"])]
            om_raw = (float(np.nanmedian(small["omega_hat"]))
                      if len(small) else np.nan)
            for _, m in sub.iterrows():
                rows.append({"channel": name, "ell": ell_i, "q": q_i,
                             "gamma_L": m["gamma_L"],
                             "lambda_coh": lam,
                             "omega_pred": aud["omega_resp"],
                             "omega_pred_ci_lo": aud["omega_resp_ci"][0],
                             "omega_pred_ci_hi": aud["omega_resp_ci"][1],
                             "omega_hat_small": om_raw,
                             "omega_hat_extrap": om0,
                             "omega_hat_ci_lo": float(om0_lo),
                             "omega_hat_ci_hi": float(om0_hi),
                             "window_points": npts,
                             "omega_hat_c08": om_checks[0.8],
                             "omega_hat_c02": om_checks[0.2],
                             "delta_pred": 2 * m["gamma"] * aud["rate_sum_resp"],
                             "delta_meas": m["delta_tilde"],
                             "delta_ci_lo": m["ci_lo"],
                             "delta_ci_hi": m["ci_hi"]})
    # SI equivalence check at the secondary location
    (e2, q2) = PARAMS[1] if len(PARAMS) > 1 else PARAMS[0]
    for name in SUITE:
        subp = dfE[(dfE["channel"] == name) & (dfE["ell"] == PARAMS[0][0])
                   & (dfE["q"] == PARAMS[0][1])]
        subs = dfE[(dfE["channel"] == name) & (dfE["ell"] == e2)
                   & (dfE["q"] == q2)]
        if len(subp) and len(subs):
            m = subp.merge(subs, on="gamma_L", suffixes=("_p", "_s"))
            equiv_rows.append({"channel": name,
                               "max_abs_diff": float(
                                   (m["delta_tilde_p"]
                                    - m["delta_tilde_s"]).abs().max())})
    if equiv_rows:
        pd.DataFrame(equiv_rows).to_csv(
            OUT_DIR / "phaseF_param_equivalence.csv", index=False)
        print("secondary-location equivalence: max |Delta difference| = "
              f"{max(r['max_abs_diff'] for r in equiv_rows):.2e}")
    dfF = pd.DataFrame(rows)
    dfF.to_csv(out_csv_F, index=False)
    # headline family statistics EXCLUDE the null coherent control, which is
    # identical to amplitude damping by construction and is an invariance
    # check rather than an independent channel case
    per8 = dfF[dfF["channel"] != "coh_diss_mix"].groupby("channel").agg(
        pred=("omega_pred", "first"), hat=("omega_hat_extrap", "first"))
    rmse8 = float(np.sqrt(np.mean((per8["hat"] - per8["pred"]) ** 2)))
    print(f"family RMSE over eight distinct channels (null control "
          f"excluded): {rmse8:.5f}")
    # joint draw-level CI on the family RMSE from the stored resamples
    if boot_store:
        # the joint RMSE interval is computed over the eight distinct
        # channels, the null duplicate excluded like every family statistic
        boot_store = {c: v for c, v in boot_store.items()
                      if c != "coh_diss_mix"}
        Bmin = min(len(v) for v in boot_store.values())
        preds = {c: float(dfF[dfF.channel == c]["omega_pred"].iloc[0])
                 for c in boot_store}
        rmse_stats = [float(np.sqrt(np.mean(
            [(boot_store[c][b] - preds[c]) ** 2 for c in boot_store])))
            for b in range(Bmin)]
        rmse_ci = np.percentile(rmse_stats, [2.5, 97.5])
        print(f"joint draw-level bootstrap on family RMSE: "
              f"({rmse_ci[0]:.4f}, {rmse_ci[1]:.4f})")
        pd.DataFrame([{"rmse_ci_lo": rmse_ci[0],
                       "rmse_ci_hi": rmse_ci[1],
                       "B": Bmin}]).to_csv(
            OUT_DIR / "phaseF_rmse_ci.csv", index=False)
    per = (dfF.groupby(["channel", "ell", "q"])
              .agg(omega_pred=("omega_pred", "first"),
                   omega_hat=("omega_hat_extrap", "first"),
                   ci_lo=("omega_hat_ci_lo", "first"),
                   ci_hi=("omega_hat_ci_hi", "first"),
                   pts=("window_points", "first")).reset_index())
    per["residual"] = per["omega_hat"] - per["omega_pred"]
    print(per.to_string(index=False, float_format=lambda v: f"{v: .4f}"))
    uniq = per.drop_duplicates(subset=["channel"]).dropna(
        subset=["omega_pred", "omega_hat"])
    rmse = float(np.sqrt(np.mean(uniq["residual"] ** 2)))
    print(f"\nomega prediction RMSE over {len(uniq)} unique channel cases "
          f"(defect window <= {DEFECT_PRIMARY}): {rmse:.4f}")
    print(f"Phase F done ({time.time() - t0:.1f}s)")
else:
    dfF = pd.read_csv(out_csv_F)

## Phase FD — direct first-order validation of $\Lambda^{\mathrm{resp}}$

The sharpest test of the central object is dimension-free. For each channel
we measure the second-moment ratio at very small per-layer strengths $h$,
form the finite-difference susceptibility on the *same* draw ensemble as
the estimator (common seed), so the comparison isolates the identity from
ensemble-to-ensemble variation, and
$\Lambda_{\mathrm{FD}}(h) = \bigl(1 - M_2(h)/M_2(0)\bigr) / (2h)$,
extrapolate $h \to 0$ by a linear fit, and compare the limit directly with
the analytically computed $\Lambda^{\mathrm{resp}}$. This validates the
susceptibility itself, independently of any normalisation, window, or
regression, and it applies unchanged to the signed pure-coherent channel,
where the finite difference must reproduce a *negative* susceptibility
(a first-order gradient enhancement).

In [ ]:
out_csv_FD = OUT_DIR / "phaseFD_firstorder.csv"
if cache_or_compute(out_csv_FD, "Phase FD"):
    print("=" * 72); print("PHASE FD -- DIRECT FINITE-DIFFERENCE VALIDATION"); print("=" * 72)
    t0 = time.time()
    H_GRID = CFG["fd_h_grid"]
    # ONE explicit draw array shared verbatim by the response audit and the
    # finite-difference reference, so the comparison isolates the first
    # order identity from any ensemble-to-ensemble sampling variation
    rng_fd = np.random.default_rng(EST_SEED)
    THETA_EST = [rng_fd.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
                 for _ in range(CFG["n_theta_est"])]
    rows = []
    for name, ch in SUITE.items():
        ctx = car.prepare_channel_context(ch, N, R, GAMMA_PROBE)
        aud = car.audit_parameter(ctx, MODEL, BETA, ELL_A, Q_A,
                                  theta_draws=THETA_EST)
        lam_fd = []
        for h in H_GRID:
            meas = cab.paired_degradation(MODEL, BETA, ELL_A, Q_A, ch, h,
                                          theta_draws=THETA_EST,
                                          shift=SHIFT, bootstrap_B=50,
                                          boot_seed=BOOT_SEED)
            lam_fd.append(meas["delta_tilde"] / (2 * h))
        lam_fd0 = float(np.polyfit(H_GRID, lam_fd, 1)[1])
        lam = aud["lambda_response"]
        rows.append({"channel": name, "case": "dissipative"
                     if name != "corr_dephase" else "protected",
                     "n_draws": CFG["n_theta_est"],
                     "lambda_resp": lam,
                     "lambda_resp_ci_lo": aud["lambda_response_ci"][0],
                     "lambda_resp_ci_hi": aud["lambda_response_ci"][1],
                     "lambda_fd_extrap": lam_fd0,
                     **{f"lambda_fd_h{h}": v for h, v in zip(H_GRID, lam_fd)},
                     "abs_err": abs(lam_fd0 - lam),
                     "rel_err": (abs(lam_fd0 - lam) / abs(lam)
                                 if abs(lam) > 1e-9 else float("nan")),
                     "bootstrap_B": aud["bootstrap_B"],
                     "bootstrap_seed": aud["bootstrap_seed"],
                     "status": aud["normalisation_status"]})
        print(f"  {name:20s} Lambda_resp {lam:+9.4f}   "
              f"Lambda_FD {lam_fd0:+9.4f}   rel err {rows[-1]['rel_err']:.2e}"
              if np.isfinite(rows[-1]['rel_err']) else
              f"Lambda_FD {lam_fd0:+9.4f}   abs residual {rows[-1]['abs_err']:.2e}")
    # signed pure-coherent case at an enlarged matched ensemble, so the
    # population susceptibility is established as stably negative with a
    # useful interval, not merely the finite-sample identity
    ch = cac.SiteZOverRotation()
    rng_sg = np.random.default_rng(EST_SEED)
    THETA_SG = [rng_sg.uniform(-DELTA_INIT, DELTA_INIT, size=(L, N))
                for _ in range(CFG["fd_signed_draws"])]
    ctx = car.prepare_channel_context(ch, N, R, GAMMA_PROBE)
    aud = car.audit_parameter(ctx, MODEL, BETA, ELL_A, Q_A,
                              theta_draws=THETA_SG)
    lam_fd = []
    for h in H_GRID:
        meas = cab.paired_degradation(MODEL, BETA, ELL_A, Q_A, ch, h,
                                      theta_draws=THETA_SG, shift=SHIFT,
                                      bootstrap_B=50, boot_seed=BOOT_SEED)
        lam_fd.append(meas["delta_tilde"] / (2 * h))
    lam_fd0 = float(np.polyfit(H_GRID, lam_fd, 1)[1])
    lam = aud["lambda_response"]
    rows.append({"channel": "site_z_overrotation", "case": "signed",
                 "n_draws": CFG["fd_signed_draws"], "lambda_resp": lam,
                 "lambda_resp_ci_lo": aud["lambda_response_ci"][0],
                 "lambda_resp_ci_hi": aud["lambda_response_ci"][1],
                 "lambda_fd_extrap": lam_fd0,
                 **{f"lambda_fd_h{h}": v for h, v in zip(H_GRID, lam_fd)},
                 "abs_err": abs(lam_fd0 - lam),
                 "rel_err": (abs(lam_fd0 - lam) / abs(lam)
                             if abs(lam) > 1e-9 else float("nan")),
                 "bootstrap_B": aud["bootstrap_B"],
                 "bootstrap_seed": aud["bootstrap_seed"],
                 "status": aud["normalisation_status"]})
    print(f"  signed ({CFG['fd_signed_draws']} draws)  "
          f"Lambda_resp {lam:+9.4f}  CI ({aud['lambda_response_ci'][0]:+.3f},"
          f"{aud['lambda_response_ci'][1]:+.3f})  Lambda_FD {lam_fd0:+9.4f}")
    dfFD = pd.DataFrame(rows)
    dfFD.to_csv(out_csv_FD, index=False)
    ok = dfFD[dfFD["case"] == "dissipative"]
    print(f"\nmax relative error over nonzero dissipative channels: "
          f"{ok['rel_err'].max():.2e}")
    prot = dfFD[dfFD["case"] == "protected"]
    print(f"protected control: |Lambda_FD| = "
          f"{abs(prot['lambda_fd_extrap'].iloc[0]):.2e}")
    print(f"Phase FD done ({time.time() - t0:.1f}s)")
else:
    dfFD = pd.read_csv(out_csv_FD)

# Phase G — predictor upgrade: pooled regression (rung 4)

The pooled small-window degradation law is refit with each candidate rate:
$\log\tilde\Delta = \alpha_0 + \alpha_1 \log(\gamma L) + \alpha_2 \log
\lambda_{\bullet}$, with $\lambda_\bullet \in \{\lambda_{\mathrm{coh}},\;
\bar r = \sum_\ell r_\ell / L\}$ (the a-priori response-weighted rate).
Points with predicted rate $\le 10^{-6}$ (the protected control) cannot enter
a log fit; they are excluded from both fits identically and validated
separately in Phase C — the honest treatment, stated here explicitly. Fits use
**unique settings only** and are reported over a tightening sequence of
accumulated-defect windows $\gamma L \lambda_{\mathrm{coh}} \leq \{0.8, 0.4,
0.2\}$: a correct first-order predictor must improve monotonically as the
window tightens, and the comparison model need not. Channel-cluster bootstrap
intervals (resampling whole channels, since CRN correlates observations
within a channel) and a leave-one-channel-out held-out prediction test
accompany the point estimates.

In [ ]:
out_csv_G = OUT_DIR / "phaseG_regression.csv"
if cache_or_compute(out_csv_G, "Phase G"):
    print("=" * 72); print("PHASE G -- PREDICTOR COMPARISON (POOLED OLS)"); print("=" * 72)
    t0 = time.time()
    d = dfF.dropna(subset=["delta_meas"]).copy()
    d = d[(d["ell"] == PARAMS[0][0]) & (d["q"] == PARAMS[0][1])]
    # the null coherent control duplicates amplitude damping by construction
    # and is excluded from every pooled statistic
    d = d[d["channel"] != "coh_diss_mix"]
    d["rbar_pred"] = d["omega_pred"] * d["lambda_coh"]
    d = d[(d["delta_meas"] > 0) & (d["gamma_L"] > 0) & (d["rbar_pred"] > 1e-6)]
    d["defect"] = d["gamma_L"] * d["lambda_coh"]
    n_excl = 0

    def _fit(dd, col):
        y = np.log(dd["delta_meas"].values)
        fitres = cab.ols_loglog(y, [np.log(dd["gamma_L"].values),
                                    np.log(dd[col].values)],
                                ["log_gL", "log_rate"])
        return fitres

    rows = []
    for c_def in (0.8, 0.4, 0.2):
        dd = d[d["defect"] <= c_def]
        for label, col in (("lambda_coh", "lambda_coh"),
                           ("response_rate", "rbar_pred")):
            fr = _fit(dd, col)
            rows.append({"window_defect": c_def, "model": label,
                         **{k: fr[k] for k in ("R2", "RMSE", "n")},
                         **fr["coef"]})
    # draw-level joint bootstrap on the R2 difference at the primary window,
    # resampling the shared CRN draw index and rebuilding every degradation
    draws = np.load(OUT_DIR / "phaseE_draws.npz")
    dd0 = d[d["defect"] <= 0.4]
    (e0, q0) = PARAMS[0]
    drng = np.random.default_rng(BOOT_SEED + 3)
    n_meas = int(dfE["n_used"].iloc[0])
    diffs_draw = []
    for _ in range(min(CFG["bootstrap_B"], 300)):
        idx = drng.integers(0, n_meas, size=n_meas)
        rows_b = []
        for _, row in dd0.iterrows():
            key = f"{row['channel']}|{e0}|{q0}|{row['gamma_L']}"
            if key + "|g0" not in draws.files:
                continue
            g0 = draws[key + "|g0"][idx]; gn = draws[key + "|gn"][idx]
            m0 = np.mean(g0)
            if m0 <= 0:
                continue
            delta_b = 1.0 - np.mean(gn) / m0
            if delta_b > 0:
                rows_b.append({"gamma_L": row["gamma_L"],
                               "lambda_coh": row["lambda_coh"],
                               "rbar_pred": row["rbar_pred"],
                               "delta_meas": delta_b})
        db = pd.DataFrame(rows_b)
        if len(db) < 10:
            continue
        try:
            ra = _fit(db, "rbar_pred")["R2"]
            rc = _fit(db, "lambda_coh")["R2"]
            diffs_draw.append(ra - rc)
        except Exception:
            pass
    if diffs_draw:
        ci_draw = np.percentile(diffs_draw, [2.5, 97.5])
        print(f"draw-level joint bootstrap on Delta-R2 (defect<=0.4): "
              f"({ci_draw[0]:.3f}, {ci_draw[1]:.3f})")
        pd.DataFrame([{"dr2_ci_lo": ci_draw[0], "dr2_ci_hi": ci_draw[1],
                       "B": len(diffs_draw)}]).to_csv(
            OUT_DIR / "phaseG_dr2_draw_ci.csv", index=False)
    # channel-cluster bootstrap at the primary window
    dd = d[d["defect"] <= 0.4]
    chans = dd["channel"].unique()
    brng = np.random.default_rng(BOOT_SEED + 2)
    r2a, diffs = [], []
    for _ in range(min(CFG["bootstrap_B"], 1000)):
        pick = brng.choice(chans, size=len(chans), replace=True)
        boot = pd.concat([dd[dd["channel"] == c] for c in pick])
        try:
            ra = _fit(boot, "rbar_pred")["R2"]
            rc = _fit(boot, "lambda_coh")["R2"]
            r2a.append(ra); diffs.append(ra - rc)
        except Exception:
            pass
    ci_r2 = np.percentile(r2a, [2.5, 97.5]) if r2a else (np.nan, np.nan)
    ci_dr = np.percentile(diffs, [2.5, 97.5]) if diffs else (np.nan, np.nan)
    # leave-one-channel-out held-out prediction (response-rate model)
    loco = []
    for c in chans:
        tr, te = dd[dd["channel"] != c], dd[dd["channel"] == c]
        fr = _fit(tr, "rbar_pred")
        b = fr["coef"]
        yh = (b["intercept"] + b["log_gL"] * np.log(te["gamma_L"])
              + b["log_rate"] * np.log(te["rbar_pred"]))
        loco.append({"channel": c,
                     "loco_rmse": float(np.sqrt(np.mean(
                         (np.log(te["delta_meas"]) - yh) ** 2)))})
    dfG = pd.DataFrame(rows)
    dfG.to_csv(out_csv_G, index=False)
    pd.DataFrame(loco).to_csv(OUT_DIR / "phaseG_loco.csv", index=False)
    # persist the cluster intervals rather than only printing them
    pd.DataFrame([{"stat": "response_R2", "lo": ci_r2[0], "hi": ci_r2[1]},
                  {"stat": "delta_R2", "lo": ci_dr[0], "hi": ci_dr[1]}]
                 ).to_csv(OUT_DIR / "phaseG_cluster_ci.csv", index=False)
    print(dfG.to_string(index=False, float_format=lambda v: f"{v: .4f}"))
    print(f"\ncluster bootstrap (defect<=0.4): response-rate R2 CI "
          f"({ci_r2[0]:.3f}, {ci_r2[1]:.3f}); Delta-R2 CI "
          f"({ci_dr[0]:.3f}, {ci_dr[1]:.3f})")
    print("LOCO held-out log-RMSE: "
          + ", ".join(f"{x['channel']}={x['loco_rmse']:.3f}" for x in loco))
    print(f"(unique settings only; protected control excluded from both fits "
          f"and validated in Phase C)")
    print(f"Phase G done ({time.time() - t0:.1f}s)")
else:
    dfG = pd.read_csv(out_csv_G)

# Phase H — layer-resolved structure and depth dependence (rung 4)

The per-slot rates $r_\ell$ expose structure invisible to any scalar: even for
restricted-isotropic channels the *response* weighting is layer-dependent
(e.g. dephasing contributes nothing at the final slot when the mode reaching
it is purely populational in the noise eigenbasis), which is why measured
alignment ratios sit below one at finite depth. The audit therefore predicts
$\omega(L)$ per channel — compared here against fresh measurements across the
depth sweep. At depth the small-box prior makes the noiseless derivative vary
strongly across draws, so the ensemble prediction is intrinsically a weighted
object: the module aggregates per-draw rates with $\partial f_0^2$ weights
(the exact first-order counterpart of the paired $M_2$ ratio) and reports the
effective sample size and a weighted-bootstrap CI, which widen precisely where
depth makes both prediction and measurement ensemble-limited. The zero-alignment protection is itself depth-limited: once the
light cone carries the gradient mode across the paired edges, the control's
alignment switches on — a prediction the depth sweep tests directly.

In [ ]:
out_csv_H = OUT_DIR / "phaseH_layer_depth.csv"
if cache_or_compute(out_csv_H, "Phase H"):
    print("=" * 72); print("PHASE H -- PER-SLOT RATES AND DEPTH DEPENDENCE"); print("=" * 72)
    t0 = time.time()
    rows = []
    gl_probe = CFG["gl_small"][0]
    for Ld in CFG["depths"]:
        model_d = cac.Brickwork(N, Ld, R)
        beta_d = cac.teacher_background(TEACHER_SEED, Ld, N)
        act_d = cab.activity_preflight(model_d, beta_d,
                                       n_draw=CFG["preflight_draws"],
                                       delta_init=DELTA_INIT, seed=PREFLIGHT_SEED)
        (ell_d, q_d) = cab.pick_parameters(act_d, k=1)[0]
        suite_d = cac.build_channel_suite(N)
        for name in ("amp_damp", "dephase", "depol", "corr_dephase"):
            ch = suite_d[name]
            aud = car.audit_channel(model_d, beta_d, ch, ell_d, q_d,
                                    n_theta=CFG["n_theta_est"],
                                    delta_init=DELTA_INIT, seed=EST_SEED,
                                    gamma_probe=GAMMA_PROBE)
            meas = cab.paired_degradation(model_d, beta_d, ell_d, q_d, ch,
                                          gl_probe / Ld,
                                          n_theta=CFG["n_theta_meas"],
                                          delta_init=DELTA_INIT, seed=MEAS_SEED,
                                          shift=SHIFT,
                                          bootstrap_B=CFG["bootstrap_B"],
                                          boot_seed=BOOT_SEED)
            om_hat = (cab.omega_hat(meas["delta_tilde"], gl_probe / Ld, Ld,
                                    aud["lambda_coh_worst"])
                      if meas["n_used"] else np.nan)
            row = {"L": Ld, "ell": ell_d, "q": q_d, "channel": name,
                   "lambda_coh": aud["lambda_coh_worst"],
                   "omega_pred": aud["omega_resp"],
                   "omega_pred_ci_lo": aud["omega_resp_ci"][0],
                   "omega_pred_ci_hi": aud["omega_resp_ci"][1],
                   "ess": aud["ess"], "omega_hat": om_hat}
            for s, v in aud["per_slot_resp"].items():
                row[f"r_slot{s}"] = v
            rows.append(row)
        print(f"  depth L={Ld} done ({time.time() - t0:.0f}s elapsed)")
    dfH = pd.DataFrame(rows)
    dfH.to_csv(out_csv_H, index=False)
    show = [c for c in dfH.columns if not c.startswith("r_slot")]
    print(dfH[show].to_string(index=False, float_format=lambda v: f"{v: .4f}"))
    print(f"Phase H done ({time.time() - t0:.1f}s)")
else:
    dfH = pd.read_csv(out_csv_H)

# Phase I — higher charge sector $r = 2$ (rung 5)

Higher charge sectors are the natural stress test: the coherence block is
far larger and the gradient mode correspondingly richer, so scalar channel
summaries are least likely to suffice. The aligned-rate audit carries over
unchanged: the pair basis simply grows to the
$r=2$ off-diagonal block. One architectural fact matters here and is worth
recording: **at the primary shallow geometry** ($L = 3$) with the localised
basis-state input, the $Z_0$ readout is $\theta$-independent at $r = 2$ to
numerical precision — the activity preflight returns a map at the numerical
floor (and raises, by design, rather than silently selecting noise). The
inactivity is geometry-specific, not universal for $r \ge 2$: residual
activity of order $10^{-4}$ reappears at greater depth. Higher-sector
validation therefore uses the fixed, seeded delocalised sector input
(`init_state="spread"`), chosen to avoid the inactive configuration.
Predictions and paired measurements then test whether generator-level
alignment remains the right organising quantity where the scalar proxy
degraded.

In [ ]:
out_csv_I = OUT_DIR / "phaseI_r2_sector.csv"
if cache_or_compute(out_csv_I, "Phase I"):
    print("=" * 72); print("PHASE I -- CHARGE SECTOR r = 2"); print("=" * 72)
    t0 = time.time()
    n2 = CFG["r2_n"]
    model_2 = cac.Brickwork(n2, L, 2, init_state="spread")
    beta_2 = cac.teacher_background(TEACHER_SEED, L, n2)
    suite_2 = cac.build_channel_suite(n2)
    names = CFG["r2_channels"] or list(suite_2)
    act_2 = cab.activity_preflight(model_2, beta_2,
                                   n_draw=CFG["preflight_draws"],
                                   delta_init=DELTA_INIT, seed=PREFLIGHT_SEED)
    (ell_2, q_2) = cab.pick_parameters(act_2, k=1)[0]
    gl_probe = CFG["gl_small"][0]
    rows = []
    for name in names:
        ch = suite_2[name]
        aud = car.audit_channel(model_2, beta_2, ch, ell_2, q_2,
                                n_theta=max(4, CFG["n_theta_est"] // 2),
                                delta_init=DELTA_INIT, seed=EST_SEED,
                                gamma_probe=GAMMA_PROBE)
        meas = cab.paired_degradation(model_2, beta_2, ell_2, q_2, ch,
                                      gl_probe / L,
                                      n_theta=CFG["n_theta_meas"],
                                      delta_init=DELTA_INIT, seed=MEAS_SEED,
                                      shift=SHIFT,
                                      bootstrap_B=CFG["bootstrap_B"],
                                      boot_seed=BOOT_SEED)
        om_hat = (cab.omega_hat(meas["delta_tilde"], gl_probe / L, L,
                                aud["lambda_coh_worst"])
                  if meas["n_used"] else np.nan)
        rows.append({"n": n2, "r": 2, "ell": ell_2, "q": q_2, "channel": name,
                     "pair_dim": len(cac.pair_basis(n2, 2)),
                     "lambda_coh": aud["lambda_coh_worst"],
                     "omega_pred": aud["omega_resp"], "omega_hat": om_hat,
                     "delta_pred": 2 * (gl_probe / L) * aud["rate_sum_resp"],
                     "delta_meas": meas["delta_tilde"]})
        print(f"  {name:14s} omega_pred={aud['omega_resp']: .4f} "
              f"omega_hat={om_hat: .4f}")
    # prediction stability across three delocalised input seeds (audit only)
    stab_rows = []
    for sseed in (cac.Brickwork.INIT_SPREAD_SEED,
                  cac.Brickwork.INIT_SPREAD_SEED + 1,
                  cac.Brickwork.INIT_SPREAD_SEED + 2):
        model_s = cac.Brickwork(n2, L, 2, init_state="spread",
                                spread_seed=sseed)
        for name in ("dephase", "corr_dephase"):
            if name not in suite_2:
                continue
            aud_s = car.audit_channel(model_s, beta_2, suite_2[name],
                                      ell_2, q_2,
                                      n_theta=max(4, CFG["n_theta_est"] // 2),
                                      delta_init=DELTA_INIT, seed=EST_SEED,
                                      gamma_probe=GAMMA_PROBE)
            stab_rows.append({"spread_seed": sseed, "channel": name,
                              "omega_pred": aud_s["omega_response"]})
    dfStab = pd.DataFrame(stab_rows)
    dfStab.to_csv(OUT_DIR / "phaseI_seed_stability.csv", index=False)
    spread = dfStab.groupby("channel")["omega_pred"].agg(["min", "max"])
    print("\nprediction stability across delocalised input seeds:")
    print(spread.to_string())
    dfI = pd.DataFrame(rows)
    dfI.to_csv(out_csv_I, index=False)
    i2x8 = dfI[dfI["channel"] != "coh_diss_mix"]
    print("sector RMSE over eight distinct channels (null control "
          f"excluded): {float(np.sqrt(np.mean((i2x8['omega_hat'] - i2x8['omega_pred']).dropna() ** 2))):.4f}")
    print(f"Phase I done ({time.time() - t0:.1f}s)")
else:
    dfI = pd.read_csv(out_csv_I)

# Phase J — estimator cost

The restricted-generator construction uses $K = d_r(d_r - 1)$ channel probes
— polynomial in the sector pair dimension — but each probe currently acts in
the full $2^n$-dimensional Hilbert space, so wall time and memory still scale
exponentially with qubit number (a sector-native implementation is the
natural next software step). The benchmark is **matched**: a full audit at
paper-level ensemble settings against the CRN measurement sweep required to
estimate the same channel–parameter alignment over the primary defect window
at the shipped ensemble size.

In [ ]:
out_csv_J = OUT_DIR / "phaseJ_cost.csv"
if cache_or_compute(out_csv_J, "Phase J"):
    print("=" * 72); print("PHASE J -- MATCHED COST BENCHMARK AND SCALING"); print("=" * 72)
    rows = []
    for n_c in CFG["cost_ns"]:
        model_c = cac.Brickwork(n_c, L, 1)
        beta_c = cac.teacher_background(TEACHER_SEED, L, n_c)
        ch = cac.build_channel_suite(n_c)["dephase"]
        t_gens, t_auds = [], []
        for _rep in range(3):
            t0 = time.time()
            L_r, pairs = car.restricted_generator(ch, n_c, 1, GAMMA_PROBE)
            t_gens.append(time.time() - t0)
            t0 = time.time()
            car.audit_channel(model_c, beta_c, ch, min(1, L - 1), 0,
                              n_theta=CFG["n_theta_est"],
                              delta_init=DELTA_INIT,
                              seed=EST_SEED, gamma_probe=GAMMA_PROBE)
            t_auds.append(time.time() - t0)
        t_gen = float(np.median(t_gens))
        t_aud = float(np.median(t_auds))
        t_aud_spread = float(max(t_auds) - min(t_auds))
        lam_c = car.lambda_coh_worst(L_r)
        gls = [gl for gl in CFG["gl_grid"] if gl * lam_c <= 0.4]
        t_sweeps = []
        for _rep in range(3):
            t0 = time.time()
            for gl in gls:
                cab.paired_degradation(model_c, beta_c, min(1, L - 1), 0, ch,
                                       gl / L, n_theta=CFG["n_theta_meas"],
                                       delta_init=DELTA_INIT, seed=MEAS_SEED,
                                       shift=SHIFT, bootstrap_B=50,
                                       boot_seed=BOOT_SEED)
            t_sweeps.append(time.time() - t0)
        t_meas = float(np.median(t_sweeps))
        t_meas_spread = float(max(t_sweeps) - min(t_sweeps))
        rows.append({"n": n_c, "sector_dim": len(cac.sector_basis(n_c, 1)),
                     "pair_dim": len(pairs),
                     "t_generator_s": t_gen, "t_full_audit_s": t_aud,
                     "t_matched_measurement_s": t_meas,
                     "t_audit_spread_s": t_aud_spread,
                     "t_sweep_spread_s": t_meas_spread,
                     "n_window_points": len(gls),
                     "audit_speedup": t_meas / t_aud if t_aud > 0 else np.nan})
        print(f"  n={n_c}: K={len(pairs)}  generator {t_gen:.2f}s  "
              f"audit({CFG['n_theta_est']} draws) {t_aud:.1f}s  "
              f"matched measurement {t_meas:.1f}s  speedup x{t_meas / t_aud:.1f}")
    dfJ = pd.DataFrame(rows)
    dfJ.to_csv(out_csv_J, index=False)
    import platform, os as _os
    print(f"environment: {platform.platform()}  python {platform.python_version()}  "
          f"numpy {np.__version__}  cpus {_os.cpu_count()}  "
          f"(timings are medians of three repetitions on a warm cache)")
    print("note: the audit needs no noisy simulation and its generator is "
          "reused across all parameters,")
    print("so the advantage compounds across parameters and noise grids; "
          "probe count is polynomial in K")
    print("but each probe currently acts in the full Hilbert space, so wall "
          "time still grows exponentially with n.")
    print("Phase J done")
else:
    dfJ = pd.read_csv(out_csv_J)

# Phase K — parameter-resolved validation map

The direct first-order identity is tested across **many genuinely distinct
parameter locations** in one circuit. At depth `pm_L` (where the growing
light cone breaks the shallow-geometry degeneracy) the activity preflight
ranks all locations, the top `pm_n_params` are audited for each channel in
`pm_channels`, and each generator-derived susceptibility is compared with a
matched-ensemble finite-difference limit at the strengths in `pm_h_grid`.
Locations inactive on the estimator ensemble are skipped with an explicit
message. Every retained point should land on the identity line at the probe
bias.

In [ ]:
out_csv_K = OUT_DIR / "phaseK_param_map.csv"
if cache_or_compute(out_csv_K, "Phase K"):
    print("=" * 72); print("PHASE K -- PARAMETER-RESOLVED VALIDATION MAP"); print("=" * 72)
    t0 = time.time()
    PM_L = CFG["pm_L"]
    PM_MODEL = cac.Brickwork(CFG["n"], PM_L, CFG["r"])
    PM_BETA = cac.teacher_background(42, PM_L, CFG["n"])
    act = cab.activity_preflight(PM_MODEL, PM_BETA,
                                 n_draw=CFG["preflight_draws"],
                                 delta_init=DELTA_INIT, seed=11)
    if isinstance(act, dict):
        cand = [k for k, _ in sorted(act.items(), key=lambda kv: -kv[1])]
    else:
        arr = np.asarray(act)
        flat = [((l, q), arr[l, q]) for l in range(arr.shape[0])
                for q in range(arr.shape[1])]
        cand = [k for k, _ in sorted(flat, key=lambda kv: -kv[1])]
    cand = cand[:CFG["pm_n_params"]]
    print("candidate locations:", cand)
    rows = []
    for name in CFG["pm_channels"]:
        ch = SUITE[name]
        ctx_k = car.prepare_channel_context(ch, CFG["n"], CFG["r"], GAMMA_PROBE)
        for (ell_k, q_k) in cand:
            aud = car.audit_parameter(ctx_k, PM_MODEL, PM_BETA, ell_k, q_k,
                                      n_theta=CFG["n_theta_est"],
                                      delta_init=DELTA_INIT, seed=EST_SEED,
                                      bootstrap_B=CFG["bootstrap_B"])
            lam = aud["lambda_response"]
            if not np.isfinite(lam):
                print(f"  {name:14s} ({ell_k},{q_k})  inactive on the "
                      "estimator ensemble, skipped")
                continue
            lam_fd = []
            for h in CFG["pm_h_grid"]:
                m = cab.paired_degradation(PM_MODEL, PM_BETA, ell_k, q_k, ch,
                                           h, n_theta=CFG["n_theta_est"],
                                           delta_init=DELTA_INIT,
                                           seed=EST_SEED, shift=np.pi / 2,
                                           bootstrap_B=50,
                                           boot_seed=MEAS_SEED + 3)
                lam_fd.append(m["delta_tilde"] / (2 * h))
            lam0 = (float(np.polyfit(CFG["pm_h_grid"], lam_fd, 1)[1])
                    if len(lam_fd) >= 2 else lam_fd[0])
            rows.append({"channel": name, "ell": ell_k, "q": q_k,
                         "lambda_resp": lam,
                         "lambda_fd_extrap": lam0,
                         **{f"h{h}": v for h, v in
                            zip(CFG["pm_h_grid"], lam_fd)},
                         "abs_err": abs(lam0 - lam),
                         "rel_err": (abs(lam0 - lam) / abs(lam)
                                     if abs(lam) > 1e-9 else float("nan")),
                         "bootstrap_B": aud["bootstrap_B"],
                         "status": aud["normalisation_status"]})
            print(f"  {name:14s} ({ell_k},{q_k})  resp {lam:+9.4f}  "
                  f"FD {lam0:+9.4f}  rel {rows[-1]['rel_err']:.2e}")
    dfK = pd.DataFrame(rows)
    dfK.to_csv(out_csv_K, index=False)
    print(f"map points: {len(dfK)}   max rel err: {dfK['rel_err'].max():.2e}")
    print(f"Phase K done ({time.time() - t0:.1f}s)")
else:
    dfK = pd.read_csv(out_csv_K)

# Publication figures

All figures are rendered from the phase CSVs (so `MODE="figures"` regenerates
them without recomputation) and written to the results `figures/` directory as
300-dpi PNG. Mathematical notation is typeset with the Computer Modern
mathtext font so symbols render as in the manuscript.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 9.5,
    "font.family": "sans-serif",
    "mathtext.fontset": "cm",            # Computer Modern maths, as typeset
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "axes.titlesize": 10, "axes.labelsize": 9.5,
    "legend.frameon": False})

# Okabe--Ito colour-blind-safe palette, keyed by channel
CHANNEL_COLOUR = {
    "amp_damp": "#0072B2", "dephase": "#D55E00", "depol": "#009E73",
    "x_err": "#CC79A7", "inhom_dephase": "#E69F00", "site_amp_damp": "#56B4E9",
    "biased_pauli": "#B8A000", "coh_diss_mix": "#999999",
    "corr_dephase": "#000000"}
CHANNEL_MARKER = {
    "amp_damp": "o", "dephase": "s", "depol": "D", "x_err": "^",
    "inhom_dephase": "v", "site_amp_damp": "P", "biased_pauli": "X",
    "coh_diss_mix": "*", "corr_dephase": "h"}
CHANNEL_LABEL = {
    "amp_damp": "Amplitude damping", "dephase": "Dephasing",
    "depol": "Depolarising", "x_err": "X error",
    "inhom_dephase": "Inhom. dephasing", "site_amp_damp": "Site-dep. damping",
    "biased_pauli": "Biased Pauli", "coh_diss_mix": "Coherent\u2013dissipative",
    "corr_dephase": "Correlated dephasing"}

def channel_handles(names):
    return [Line2D([], [], ls="none", marker=CHANNEL_MARKER[c],
                   color=CHANNEL_COLOUR[c], ms=6, label=CHANNEL_LABEL[c])
            for c in names]

def bottom_legend(fig, names, ncol=5):
    # anchor the legend's TOP edge at the bottom of the figure canvas so it
    # always sits fully below the axes and axis labels; bbox_inches="tight"
    # then expands the saved canvas to include it
    fig.legend(handles=channel_handles(names), loc="upper center",
               bbox_to_anchor=(0.5, 0.0), ncol=ncol, fontsize=8,
               handletextpad=0.35, columnspacing=1.1)

def save_fig(fig, stem):
    fig.savefig(FIG_DIR / f"{stem}.png", bbox_inches="tight", dpi=300)
    print(f"[figure] {stem}.png written (300 dpi)")

### Figure 1 — recovery and control

(a) Isotropic recovery: $\lambda_{\mathrm{vis}}^{\mathrm{op}}$ against
$\lambda_{\mathrm{coh}}$ for the four restricted-isotropic channels lies on
the diagonal to four significant figures. (b) The zero-alignment control,
parameter-resolved: the generator-level prediction separates protected from
exposed parameters, confirmed by the paired measurements.

In [ ]:
b = pd.read_csv(OUT_DIR / "phaseB_isotropic_recovery.csv")
c = pd.read_csv(OUT_DIR / "phaseC_zero_alignment.csv")
c = c.sort_values(["protected_pred", "ell"], ascending=[False, True]).reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.0, 3.6))

# (a) isotropic recovery: points on the identity line; legend in the empty
#     upper-left triangle above the diagonal
lim = [0, 1.15 * b["lambda_coh"].max()]
ax1.plot(lim, lim, "-", color="0.78", lw=1, zorder=0)
for _, row in b.iterrows():
    ax1.scatter(row["lambda_coh"], row["lambda_vis_op"], s=52,
                color=CHANNEL_COLOUR[row["channel"]],
                marker=CHANNEL_MARKER[row["channel"]],
                label=CHANNEL_LABEL[row["channel"]], zorder=3)
ax1.set_xlim(lim); ax1.set_ylim(lim)
ax1.set_xlabel(r"Worst-case coherence rate $\lambda_{\mathrm{coh}}$")
ax1.set_ylabel(r"Mode-contraction rate $\lambda_{\mathrm{mode}}$")
ax1.set_title("(a) Restricted-isotropic recovery")
ax1.legend(loc="upper left", fontsize=8)

# (b) zero-alignment control: protected parameters first (small bars on the
#     left), exposed parameters last, so the upper-left region stays empty
#     for the legend
gl0 = CFG["gl_small"][0]
c = c.sort_values("omega_op").reset_index(drop=True)
om_meas = c["delta_meas"] / (2 * gl0 * c["lambda_coh"])
xs = np.arange(len(c)); w = 0.22
ax2.axhline(0.0, color="0.75", lw=1, zorder=0)
ax2.plot(xs - w, c["omega_op"], ls="none", marker="s", ms=8,
         color="#999999", label=r"Mode ratio $\omega^{\mathrm{mode}}$",
         zorder=3)
ax2.plot(xs, c["omega_resp_pred"], ls="none", marker="o", ms=8,
         color="#0072B2", label=r"Response prediction $\omega^{\mathrm{resp}}$",
         zorder=3)
ax2.plot(xs + w, om_meas, ls="none", marker="D", ms=7, mfc="white",
         color="#D55E00", label=r"Measured $\hat{\omega}$", zorder=3)
ax2.set_xticks(xs)
ax2.set_xticklabels([rf"$({int(r.ell)},{int(r.q)})$" for r in c.itertuples()],
                    fontsize=8.5)
ax2.set_ylim(-0.08, max(1.0, 1.45 * max(c["omega_op"].max(),
                                         om_meas.max(), 0.01)))
ax2.set_xlabel(r"Parameter $(\ell, q)$")
ax2.set_ylabel(r"Alignment ratio $\omega$")
ax2.set_title("(b) Zero-alignment control, parameter-resolved")
ax2.legend(loc="upper left", fontsize=8)

fig.tight_layout(w_pad=2.0)
save_fig(fig, "fig1_recovery_control")
plt.close(fig)

### Figure 2 — central validation: predicted vs measured alignment

Every (channel, parameter) point: a-priori $\omega_{\mathrm{pred}}$ from the
audit against measured $\hat\omega$ from the small-noise window of fresh CRN
data. The diagonal is the claim of the study; the protected control sits at
the origin.

In [ ]:
f = pd.read_csv(OUT_DIR / "phaseF_validation.csv")
per = (f.groupby(["channel", "ell", "q"])
        .agg(omega_pred=("omega_pred", "first"),
             ci_lo=("omega_pred_ci_lo", "first"),
             ci_hi=("omega_pred_ci_hi", "first"),
             omega_hat=("omega_hat_extrap", "first")).reset_index()
        .dropna(subset=["omega_pred", "omega_hat"])
        .drop_duplicates(subset=["channel"]))   # unique channel cases

fd = pd.read_csv(OUT_DIR / "phaseFD_firstorder.csv")
diss = fd[fd["case"] == "dissipative"]; prot = fd[fd["case"] == "protected"]
sgn = fd[fd["case"] == "signed"].iloc[0]
fig, (ax, axb) = plt.subplots(1, 2, figsize=(9.2, 4.4),
                              gridspec_kw={"width_ratios": [1.55, 1]})
lim_hi = max(diss["lambda_resp"].max(), diss["lambda_fd_extrap"].max()) * 1.1
ax.plot([0, lim_hi], [0, lim_hi], "-", color="0.78", lw=1, zorder=0)
for _, row in pd.concat([diss, prot]).iterrows():
    ax.scatter(row["lambda_resp"], row["lambda_fd_extrap"], s=52,
               color=CHANNEL_COLOUR[row["channel"]],
               marker=CHANNEL_MARKER[row["channel"]], zorder=3)
ax.text(0.96, 0.13, rf"max rel.\ err.\ $= {diss['rel_err'].max():.1e}$",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=9)
ax.text(0.96, 0.05, r"protected control $|\Lambda_{\mathrm{FD}}| = "
        rf"{abs(prot['lambda_fd_extrap'].iloc[0]):.1e}$",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=9)
ax.set_xlim(-0.8, lim_hi); ax.set_ylim(-0.8, lim_hi)
ax.set_xlabel(r"CohAlign susceptibility $\Lambda^{\mathrm{resp}}$ (generator derived)")
ax.set_ylabel(r"Finite-difference limit $\Lambda_{\mathrm{FD}}$")
ax.set_title("(a) Dissipative suite and protected control")
axb.axhline(0.0, color="0.75", lw=1, zorder=0)
axb.errorbar([0.0], [sgn["lambda_resp"]],
             yerr=[[sgn["lambda_resp"] - sgn["lambda_resp_ci_lo"]],
                   [sgn["lambda_resp_ci_hi"] - sgn["lambda_resp"]]],
             fmt="o", color="#661100", capsize=4, ms=6,
             label=r"$\Lambda^{\mathrm{resp}}$ with bootstrap CI", zorder=3)
axb.plot([0.0], [sgn["lambda_fd_extrap"]], marker="_", ms=22, mew=2.2,
         ls="none", color="#0072B2",
         label=r"$\Lambda_{\mathrm{FD}}$ (matched draws)", zorder=4)
axb.set_xlim(-0.6, 0.6); axb.set_xticks([])
axb.set_ylabel(r"Susceptibility $\Lambda$")
axb.set_title(f"(b) Signed coherent case ({int(sgn['n_draws'])} draws)")
axb.legend(loc="lower center", fontsize=8, frameon=False)
handles = [Line2D([0], [0], ls="none", marker=CHANNEL_MARKER[c],
                  color=CHANNEL_COLOUR[c], ms=7, label=CHANNEL_LABEL[c])
           for c in pd.concat([diss, prot])["channel"]]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.0),
           ncol=3, fontsize=8, frameon=False)
save_fig(fig, "fig2_first_order_validation")

fig, ax = plt.subplots(figsize=(4.8, 4.5))
lim = [-0.06, 1.12 * max(per["omega_pred"].max(), per["omega_hat"].max(), 1.0)]
ax.plot(lim, lim, "-", color="0.78", lw=1, zorder=0)
for _, row in per.iterrows():
    xerr = np.array([[max(row["omega_pred"] - row["ci_lo"], 0.0)],
                     [max(row["ci_hi"] - row["omega_pred"], 0.0)]])
    ax.errorbar(row["omega_pred"], row["omega_hat"], xerr=xerr,
                fmt=CHANNEL_MARKER[row["channel"]], ms=6.5,
                color=CHANNEL_COLOUR[row["channel"]],
                elinewidth=0.8, capsize=2, zorder=3)
per = per[per.index.get_level_values("channel") != "coh_diss_mix"] \
    if "channel" in getattr(per.index, "names", []) else \
    per[per["channel"] != "coh_diss_mix"] if "channel" in per.columns else per
rmse = float(np.sqrt(np.mean((per["omega_hat"] - per["omega_pred"]) ** 2)))
# annotation in the empty lower-right triangle, well clear of the diagonal
ax.text(0.96, 0.10, rf"RMSE $= {rmse:.3f}$", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=9)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel(r"Predicted alignment ratio $\omega^{\mathrm{resp}}$")
ax.set_ylabel(r"Measured alignment ratio $\hat{\omega}$ (defect-window intercept)")
ax.set_title("Predicted alignment against measurement")
bottom_legend(fig, list(dict.fromkeys(per["channel"])), ncol=3)
save_fig(fig, "figS1_alignment_ratio")
plt.close(fig)

### Figure 3 — first-order prediction across the noise grid, and the
predictor comparison

(a) $\tilde\Delta_{\mathrm{pred}} = 2\gamma\sum_\ell r_\ell$ against
$\tilde\Delta_{\mathrm{meas}}$ for every channel, parameter and noise
strength (log–log; departures at large $\gamma L$ are the expected
higher-order corrections). (b) Pooled OLS: $R^2$ and RMSE for the scalar
$\lambda_{\mathrm{coh}}$ predictor against the a-priori aligned rate.

In [ ]:
f = pd.read_csv(OUT_DIR / "phaseF_validation.csv")
g = pd.read_csv(OUT_DIR / "phaseG_regression.csv")
if "window_defect" in g.columns:
    g = g[np.isclose(g["window_defect"], 0.4)].reset_index(drop=True)
elif "window" in g.columns:
    g = g[g["window"] != "full_grid"].reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.2, 3.8),
                               gridspec_kw={"width_ratios": [1.3, 1]})

# (a) first-order prediction across the grid; legend lives below the figure
d = f[(f["delta_pred"] > 1e-9) & (f["delta_meas"] > 1e-9)]
for _, row in d.iterrows():
    ax1.loglog(row["delta_pred"], row["delta_meas"], ls="none",
               color=CHANNEL_COLOUR[row["channel"]],
               marker=CHANNEL_MARKER[row["channel"]], ms=5.5, zorder=3)
lim = [min(d["delta_pred"].min(), d["delta_meas"].min()) * 0.7,
       max(d["delta_pred"].max(), d["delta_meas"].max()) * 1.4]
ax1.plot(lim, lim, "-", color="0.78", lw=1, zorder=0)
ax1.set_xlim(lim); ax1.set_ylim(lim)
ax1.set_xlabel(r"Predicted degradation $\tilde{\Delta}_{\mathrm{pred}} = 2\gamma \sum_{\ell} r_{\ell}$")
ax1.set_ylabel(r"Measured degradation $\tilde{\Delta}_{\mathrm{meas}}$")
ax1.set_title("(a) First-order prediction, all settings")

# (b) perturbative boundary: measurement-to-prediction ratio against the
#     accumulated defect, one point per setting, with the first-order line
d2 = d[d["channel"] != "coh_diss_mix"].copy()
d2["ratio"] = d2["delta_meas"] / d2["delta_pred"]
for _, row in d2.iterrows():
    ax2.semilogx(row["delta_pred"], row["ratio"], ls="none",
                 color=CHANNEL_COLOUR[row["channel"]],
                 marker=CHANNEL_MARKER[row["channel"]], ms=5, zorder=3)
ax2.axhline(1.0, color="0.7", lw=1, zorder=0)
ax2.set_xlabel(r"Predicted degradation $2\gamma\Lambda^{\mathrm{resp}}$")
ax2.set_ylabel(r"Ratio $\tilde{\Delta}_{\mathrm{meas}} / \tilde{\Delta}_{\mathrm{pred}}$")
ax2.set_title("(b) Boundary of the first-order regime")
ax2.set_ylim(0.0, 1.15)
if False:
    xs = np.arange(len(g))
    bars = ax2.bar(xs, g["R2"], 0.55, color=["#999999", "#0072B2"])

fig.tight_layout(w_pad=2.0)
bottom_legend(fig, list(dict.fromkeys(d["channel"])), ncol=4)
save_fig(fig, "fig3_prediction_and_regression")
plt.close(fig)

### Figure 4 — layer-resolved rates and depth dependence

(a) Per-slot response-weighted rates $r_\ell$ at the deepest audited
architecture — the structure a scalar cannot express. (b) Predicted and
measured alignment ratio against depth.

In [ ]:
h = pd.read_csv(OUT_DIR / "phaseH_layer_depth.csv")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.2, 3.7))
h_names = list(dict.fromkeys(h["channel"]))

# (a) per-slot rates at the deepest audited architecture
Lmax = int(h["L"].max())
hh = h[h["L"] == Lmax]
slot_cols = sorted([col for col in h.columns if col.startswith("r_slot")],
                   key=lambda s: int(s.replace("r_slot", "")))
for _, row in hh.iterrows():
    vals = [row[col] for col in slot_cols if np.isfinite(row.get(col, np.nan))]
    ax1.plot(range(len(vals)), vals, "-", marker=CHANNEL_MARKER[row["channel"]],
             ms=5, color=CHANNEL_COLOUR[row["channel"]], lw=1.4)
ax1.set_xticks(range(Lmax))
ax1.set_xlabel(r"Noise slot $\ell$")
ax1.set_ylabel(r"Per-slot response rate $r_{\ell}$")
ax1.set_title(rf"(a) Layer-resolved rates at $L = {Lmax}$")

# (b) alignment against depth: channel identity by colour (shared legend
#     below), estimator/measurement by line style (small legend inside,
#     anchored to the upper-right corner clear of the data)
for name, sub in h.groupby("channel"):
    sub = sub.sort_values("L")
    if {"omega_pred_ci_lo", "omega_pred_ci_hi"} <= set(sub.columns):
        yerr = np.vstack([(sub["omega_pred"] - sub["omega_pred_ci_lo"]).clip(lower=0),
                          (sub["omega_pred_ci_hi"] - sub["omega_pred"]).clip(lower=0)])
        ax2.errorbar(sub["L"], sub["omega_pred"], yerr=yerr, fmt="-",
                     marker=CHANNEL_MARKER[name], ms=5,
                     color=CHANNEL_COLOUR[name], lw=1.4,
                     elinewidth=0.8, capsize=2)
    else:
        ax2.plot(sub["L"], sub["omega_pred"], "-",
                 marker=CHANNEL_MARKER[name], ms=5,
                 color=CHANNEL_COLOUR[name], lw=1.4)
    ax2.plot(sub["L"], sub["omega_hat"], "--",
             marker=CHANNEL_MARKER[name], ms=5, mfc="white",
             color=CHANNEL_COLOUR[name], lw=1.2, alpha=0.85)
style_handles = [Line2D([], [], color="0.25", ls="-", marker=".",
                        label="Predicted"),
                 Line2D([], [], color="0.25", ls="--", marker=".",
                        mfc="white", label="Measured")]
ymax = np.nanmax([h["omega_pred"].max(), h["omega_hat"].max()])
ax2.set_ylim(-0.04, 1.32 * max(ymax, 1.0))
ax2.set_ylim(-0.05, 1.40)
ax2.legend(handles=style_handles, loc="upper center", fontsize=8, ncol=2,
           frameon=False)
depths = sorted(h["L"].unique())
plabels = []
for Ld in depths:
    sub = h[h["L"] == Ld]
    plabels.append(rf"{Ld}" + "\n" +
                   rf"$({int(sub['ell'].iloc[0])},{int(sub['q'].iloc[0])})$")
ax2.set_xticks(depths)
ax2.set_xticklabels(plabels, fontsize=8.5)
ax2.set_xlabel(r"Depth $L$ (audited parameter)")
ax2.set_ylabel(r"Alignment ratio $\omega$")
ax2.set_title("(b) Alignment against depth")

fig.tight_layout(w_pad=2.0)
bottom_legend(fig, h_names, ncol=4)
save_fig(fig, "fig4_layer_resolved_depth")
plt.close(fig)

### Figure 5 — higher sector and audit cost

(a) Predicted vs measured alignment in the $r=2$ sector. (b) Estimator cost
against the pair dimension $K$.

In [ ]:
i2 = pd.read_csv(OUT_DIR / "phaseI_r2_sector.csv")
j = pd.read_csv(OUT_DIR / "phaseJ_cost.csv")

fig, ax1 = plt.subplots(figsize=(5.2, 4.2))

# higher-sector validation; channel legend shared below the figure
lim = [-0.05, 1.15 * max(1.0, i2["omega_pred"].max(),
                         i2["omega_hat"].fillna(0).max())]
ax1.plot(lim, lim, "-", color="0.78", lw=1, zorder=0)
for _, row in i2.iterrows():
    ax1.scatter(row["omega_pred"], row["omega_hat"], s=55,
                color=CHANNEL_COLOUR[row["channel"]],
                marker=CHANNEL_MARKER[row["channel"]], zorder=3)
ax1.set_xlim(lim); ax1.set_ylim(lim)
i2x = i2[i2["channel"] != "coh_diss_mix"]  # null duplicate excluded
rmse2 = float(np.sqrt(np.mean((i2x["omega_hat"] - i2x["omega_pred"]) ** 2)))
ax1.text(0.96, 0.08, rf"RMSE $= {rmse2:.3f}$", transform=ax1.transAxes,
         ha="right", va="bottom", fontsize=9)
ax1.set_xlabel(r"Predicted alignment ratio $\omega^{\mathrm{resp}}$")
ax1.set_ylabel(r"Measured alignment ratio $\hat{\omega}$")
ax1.set_title(rf"Charge sector $r = 2$  ($n = {int(i2['n'].iloc[0])}$)")

save_fig(fig, "fig5_r2_sector")

# ---- fig6: parameter-resolved validation map ----
dK = pd.read_csv(OUT_DIR / "phaseK_param_map.csv")
_COLK = {"dephase": "#D55E00", "corr_dephase": "#000000",
         "amp_damp": "#0072B2"}
_MARKK = {"dephase": "s", "corr_dephase": "h", "amp_damp": "o"}
_LABK = {"dephase": "Dephasing", "corr_dephase": "Correlated dephasing",
         "amp_damp": "Amplitude damping"}
fig, ax = plt.subplots(figsize=(5.1, 4.4))
_lim = [0, 1.12 * max(dK["lambda_resp"].max(), dK["lambda_fd_extrap"].max())]
ax.plot(_lim, _lim, "-", color="0.78", lw=1, zorder=0)
for _ch, _sub in dK.groupby("channel"):
    ax.scatter(_sub["lambda_resp"], _sub["lambda_fd_extrap"], s=58,
               facecolors="none", edgecolors=_COLK.get(_ch, "#333333"),
               linewidths=1.6, marker=_MARKK.get(_ch, "o"), zorder=3,
               label=_LABK.get(_ch, _ch))
ax.text(0.96, 0.06,
        rf"max rel. err. $= {dK['rel_err'].max():.1e}$ (probe bias)",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=8.6)
ax.set_xlim(_lim); ax.set_ylim(_lim)
ax.set_xlabel(r"CohAlign susceptibility $\Lambda^{\mathrm{resp}}$ (generator derived)")
ax.set_ylabel(r"Finite-difference limit $\Lambda_{\mathrm{FD}}$")
ax.set_title(f"Parameter-resolved validation at depth {CFG['pm_L']}")
ax.legend(loc="upper left", fontsize=8.6, frameon=False)
save_fig(fig, "fig6_param_map")

fig, ax2 = plt.subplots(figsize=(5.0, 3.9))
# audit cost; two-entry legend in the empty upper-left region
ax2.loglog(j["pair_dim"], j["t_generator_s"], "-o", ms=5.5,
           color="#0072B2", label="Restricted generator")
ax2.loglog(j["pair_dim"], j["t_full_audit_s"], "-s", ms=5.5,
           color="#D55E00", label="Full audit")
if "t_matched_measurement_s" in j.columns:
    ax2.loglog(j["pair_dim"], j["t_matched_measurement_s"], "-^", ms=5.5,
               color="#009E73", label="Matched finite-noise sweep")
ax2.set_xlabel(r"Pair dimension $K = d_r (d_r - 1)$")
ax2.set_ylabel("Wall-clock time (s)")
ax2.set_title("Audit cost, full-ensemble benchmark")
ax2.legend(loc="upper left", fontsize=8)

fig.tight_layout(w_pad=2.0)
bottom_legend(fig, list(dict.fromkeys(i2["channel"])), ncol=5)
save_fig(fig, "figS2_cost_benchmark")
plt.close(fig)

# Worked example — auditing a user-supplied channel

The audit in six lines: define a channel (here a device-like asymmetric
$T_1/T_2$ model with site-dependent dephasing bias), call `audit_channel`,
read off the rates and the verdict. This is the intended usage pattern for
hardware-aware ansatz placement: parameters whose modes align with
slowly-contracting directions of the *measured* device generator retain
trainability longest.

In [ ]:
class DeviceLikeChannel(cac.NoiseChannel):
    '''Asymmetric T1/T2: amplitude damping at gamma plus extra dephasing at
    2*gamma on odd sites only (a crude cross-talk-free device caricature).'''
    name = "device_like"
    def apply(self, M, gamma, n):
        Ks_ad = [np.array([[1, 0], [0, np.sqrt(1 - gamma)]], dtype=complex),
                 np.array([[0, np.sqrt(gamma)], [0, 0]], dtype=complex)]
        for q in range(n):
            M = cac._apply_1q_kraus(M, Ks_ad, q, n)
        g2 = min(2 * gamma, 1.0)
        Ks_dz = [np.sqrt(1 - g2) * cac.I2, np.sqrt(g2) * cac.PAULI_Z]
        for q in range(1, n, 2):
            M = cac._apply_1q_kraus(M, Ks_dz, q, n)
        return M

device = DeviceLikeChannel()
report = car.audit_channel(MODEL, BETA, device, ELL_A, Q_A,
                           n_theta=CFG["n_theta_est"], delta_init=DELTA_INIT,
                           seed=EST_SEED, gamma_probe=GAMMA_PROBE)
print("audit report for the user-supplied channel")
print("-" * 46)
for key in ("channel", "n", "L", "r", "ell_i", "q_i", "lambda_coh_worst",
            "lambda_coh_typical", "lambda_vis_op", "omega_op",
            "rate_sum_resp", "omega_resp", "wall_seconds"):
    v = report[key]
    print(f"  {key:20s} {v:.4f}" if isinstance(v, float) else f"  {key:20s} {v}")
print(f"  per-slot rates      "
      + ", ".join(f"r_{k}={v:.3f}" for k, v in sorted(report["per_slot_resp"].items())))
print("-" * 46)
print(f"verdict: parameter ({ELL_A},{Q_A}) loses gradient signal at "
      f"{report['omega_resp']:.2f} x the worst-case sector rate under this "
      f"channel;\n         predicted relative degradation per unit gamma*L: "
      f"{2 * report['rate_sum_resp'] / L:.3f}")

# Outputs summary and run metadata

In [ ]:
print("=" * 76)
print("COHALIGN OUTPUTS SUMMARY")
print("=" * 76)
for p in sorted(OUT_DIR.glob("phase*.csv")):
    df = pd.read_csv(p)
    print(f"  {p.name:38s} {len(df):5d} rows x {len(df.columns):2d} cols")
for p in sorted(FIG_DIR.glob("*.png")):
    print(f"  figures/{p.name}")
meta = {
    "project": "CohAlign",
    "mode": MODE, "config": {k: (list(v) if isinstance(v, tuple) else v)
                              for k, v in CFG.items()},
    "delta_init": DELTA_INIT, "gamma_probe": GAMMA_PROBE,
    "teacher_seed": TEACHER_SEED,
    "seeds": {"preflight": PREFLIGHT_SEED, "estimator": EST_SEED,
              "measurement": MEAS_SEED, "bootstrap": BOOT_SEED},
    "python": platform.python_version(),
    "numpy": np.__version__, "pandas": pd.__version__,
    "wall_seconds_total": round(time.time() - t_notebook_start, 1),
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with open(OUT_DIR / "run_metadata.json", "w") as fh:
    json.dump(meta, fh, indent=2)
import hashlib, platform, glob
def _sha(path):
    try:
        return hashlib.sha256(open(path, "rb").read()).hexdigest()[:16]
    except Exception:
        return "absent"
src_hashes = {m: _sha(PKG_DIR / f"{m}.py")
              for m in ("cohalign_core", "cohalign_rates", "cohalign_bench")}
try:
    _ram_gb = round(int([l for l in open("/proc/meminfo")
                         if l.startswith("MemTotal")][0].split()[1]) / 1e6, 1)
except Exception:
    _ram_gb = None
_nb_candidates = sorted(glob.glob("cohalign_pipeline*.ipynb"))
# never search a mounted drive recursively for the notebook hash; a FUSE
# walk over a large Drive can run for many minutes and looks like a hang
out_hashes = {p.name: _sha(p) for p in sorted(OUT_DIR.glob("*.csv"))}
_figdir = OUT_DIR / "figures"
fig_hashes = ({p.name: _sha(p) for p in sorted(_figdir.glob("*.png"))}
              if _figdir.exists() else {})
import matplotlib as _mpl
manifest = {"mode": MODE, "config": {k: str(v) for k, v in CFG.items()},
            "module_sha256_16": src_hashes,
            "notebook_sha256_16": (_sha(_nb_candidates[0])
                                   if _nb_candidates else "not_located"),
            "output_csv_sha256_16": out_hashes,
            "figure_sha256_16": fig_hashes,
            "operating_system": f"{platform.system()} {platform.release()}",
            "processor": platform.processor() or platform.machine(),
            "logical_cores": os.cpu_count(), "ram_gb": _ram_gb,
            "python_version": platform.python_version(),
            "numpy_version": np.__version__,
            "pandas_version": pd.__version__,
            "matplotlib_version": _mpl.__version__,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
if MODE == "figures":
    # a figures-mode session renders from cached CSVs and must never
    # overwrite the record of the run that produced them
    with open(OUT_DIR / "render_manifest.json", "w") as fh:
        json.dump(manifest, fh, indent=1)
    print("figures mode: render_manifest.json written; the paper-run "
          "manifest is left untouched")
else:
    with open(OUT_DIR / f"{MODE}_run_manifest.json", "w") as fh:
        json.dump(manifest, fh, indent=1)
    print(f"{MODE}_run_manifest.json written")
# one machine-readable summary of every headline number in the paper
try:
    _fd = pd.read_csv(OUT_DIR / "phaseFD_firstorder.csv")
    _f = pd.read_csv(OUT_DIR / "phaseF_validation.csv")
    _g = pd.read_csv(OUT_DIR / "phaseG_regression.csv")
    _i = pd.read_csv(OUT_DIR / "phaseI_r2_sector.csv")
    _p8 = _f[_f.channel != "coh_diss_mix"].groupby("channel").agg(
        pred=("omega_pred", "first"), hat=("omega_hat_extrap", "first"))
    _sg = _fd[_fd["case"] == "signed"].iloc[0]
    _i8 = _i[_i.channel != "coh_diss_mix"]
    headline = {
        "family_rmse_8": float(np.sqrt(np.mean((_p8.hat - _p8.pred) ** 2))),
        "fd_max_rel_err_dissipative": float(
            _fd[_fd["case"] == "dissipative"]["rel_err"].max()),
        "fd_protected_abs_residual": float(
            _fd[_fd["case"] == "protected"]["lambda_fd_extrap"].abs().iloc[0]),
        "signed_lambda_resp": float(_sg["lambda_resp"]),
        "signed_ci": [float(_sg["lambda_resp_ci_lo"]),
                      float(_sg["lambda_resp_ci_hi"])],
        "signed_lambda_fd": float(_sg["lambda_fd_extrap"]),
        "signed_n_draws": int(_sg["n_draws"]),
        "regression": _g.to_dict("records"),
        "sector_rmse_8": float(np.sqrt(np.mean(
            (_i8["omega_hat"] - _i8["omega_pred"]).dropna() ** 2))),
    }
    with open(OUT_DIR / "headline_numbers.json", "w") as fh:
        json.dump(headline, fh, indent=1)
    print("headline_numbers.json written")
except Exception as e:
    print("headline summary skipped:", e)

# archive the exact module sources next to the results (repo/Zenodo deposit)
SRC_DIR = OUT_DIR / "cohalign_src"
SRC_DIR.mkdir(exist_ok=True)
for mod in PKG_DIR.glob("cohalign_*.py"):
    (SRC_DIR / mod.name).write_text(mod.read_text())
print(f"module sources archived -> {SRC_DIR}")
print(f"\nrun_metadata.json written | total wall time "
      f"{meta['wall_seconds_total']:.1f}s | MODE={MODE}")

# Reproducibility, repository layout, and data availability

**Repository layout materialised by this notebook.**
`cohalign/` — the three library modules in the runtime, written verbatim from
the source cells above (so notebook and package cannot drift) and archived
alongside the results as `cohalign_src/`; the results directory (Google Drive
on Colab, `cohalign_outputs/` locally) — one CSV per phase plus
`run_metadata.json`; `figures/` within it — all publication figures as
300-dpi PNG. The notebook is idempotent: phase
CSVs are cached and re-used unless `FORCE_RERUN = True`.

**Determinism.** All randomness flows through explicit
`numpy.random.default_rng` seeds recorded in `run_metadata.json`; common
random numbers pair every noisy measurement with its noiseless baseline and
share draws across channels and noise strengths.

**Modes.** `smoke` (minutes; reduced grids at $n=6$) verifies the full
pipeline end to end; `paper` regenerates the manuscript numbers at $n=8$;
`figures` re-renders figures from cached CSVs.

**Data availability (draft statement for submission).** All data underlying
the study are generated deterministically by this notebook; the archived CSVs
and figures, together with the exact notebook and package sources, are
deposited at the project repository and a versioned Zenodo release
(DOI: *to be minted at submission*). No third-party or archived experimental
data are used.

**Scope note.** The response-weighted estimator is exact at first order in
$\gamma$; departures at $\gamma L \gtrsim 0.2$ visible in Figure 3(a) are the
expected higher-order corrections and are reported, not fitted away. The
quadratic estimator $\lambda_{\mathrm{vis}}^{\mathrm{op}}$ is state-independent
and coincides with the response-weighted rate on the restricted-isotropic
family and on fully protected modes; where the two differ (structured noise at
partially-overlapping modes), Phase F quantifies the difference — the
response-weighted rate is the recommended audit output.